In [ ]:
!pip -q install pandas numpy scipy scikit-learn faker transformers accelerate gradio


In [ ]:
# ------------------------------------------------------------
# DASHBOARD MODE
#   True  -> terminal prompts / step-by-step demo cells are skipped,
#            use the web dashboard in the last cells (Runtime > Run all).
#   False -> original behaviour (input() prompts, cell-by-cell flow).
# ------------------------------------------------------------
DASHBOARD_ONLY = True

import pandas as pd
import numpy as np
import json
import re
import os
from collections import Counter
from faker import Faker

try:
    from google.colab import files
except ImportError:
    files = None

fake = Faker()


In [ ]:
if not DASHBOARD_ONLY:
    try:
        from google.colab import files
    except ImportError:
        files = None

    import pandas as pd
    import json
    import os

    print("=" * 60)
    print("        AI SYNTHETIC DATASET GENERATOR (SCHEMA INPUT)")
    print("=" * 60)

    print("""
    Choose your schema input method:

    [1] Upload Schema File (.json, .csv, .txt, .md)
    [2] Paste Schema JSON Directly (Press Enter for default schema)
    """)

    choice = input("Enter 1 or 2 (Default: 2): ").strip()
    if not choice or choice not in ["1", "2"]:
        choice = "2"

    # ============================================================
    # OPTION 1: UPLOAD SCHEMA FILE
    # ============================================================
    if choice == "1":
        if files is not None:
            print("\nPlease upload your schema file...")
            uploaded = files.upload()

            if len(uploaded) == 0:
                raise ValueError("No file uploaded.")

            filename = list(uploaded.keys())[0]
        else:
            filename = input("Enter path to schema file: ").strip()
            if not filename or not os.path.exists(filename):
                raise ValueError(f"File not found: {filename}")

        print(f"\nReading: {filename}")
        extension = os.path.splitext(filename)[1].lower()

        if extension == ".json":
            with open(filename, "r", encoding="utf-8") as f:
                schema = json.load(f)

        elif extension in [".txt", ".md"]:
            with open(filename, "r", encoding="utf-8") as f:
                raw_text = f.read().strip()
                try:
                    schema = json.loads(raw_text)
                except Exception:
                    schema = raw_text

        elif extension == ".csv":
            schema_df = pd.read_csv(filename)
            schema = schema_df.to_dict(orient="records")

        else:
            raise ValueError("Schema file must be .json, .csv, .txt, or .md")

        print("\n✓ Schema uploaded successfully.")

    # ============================================================
    # OPTION 2: PASTE SCHEMA JSON DIRECTLY
    # ============================================================
    elif choice == "2":
        print("\nPaste your JSON schema below (or press Enter to use default schema):")
        pasted_schema = input("Schema JSON: ").strip()

        if not pasted_schema:
            print("✓ Using standard default customer schema...")
            schema = {
                "dataset_name": "customer_data",
                "columns": [
                    {"name": "customer_id", "type": "identifier", "format": "CUST-#####"},
                    {"name": "full_name", "type": "name"},
                    {"name": "email", "type": "email"},
                    {"name": "age", "type": "integer", "minimum": 18, "maximum": 75},
                    {"name": "gender", "type": "categorical", "allowed_values": ["Male", "Female", "Non-Binary", "Other"]},
                    {"name": "subscription", "type": "categorical", "allowed_values": ["Free", "Basic", "Premium"]},
                    {"name": "annual_income", "type": "float", "minimum": 20000.0, "maximum": 150000.0},
                    {"name": "is_active", "type": "boolean"}
                ]
            }
        else:
            schema = json.loads(pasted_schema)

        print("\n✓ Schema loaded successfully.")

    print("\nLoaded Schema Definition:")
    print(json.dumps(schema, indent=2, default=str) if isinstance(schema, (dict, list)) else schema)


In [ ]:
import pandas as pd
import numpy as np
import json
import re


# ============================================================
# 1. COLUMN TYPE DETECTION
# ============================================================

def detect_column_type(series):

    # Remove missing values for analysis
    s = series.dropna()

    if len(s) == 0:
        return "unknown"

    # Boolean
    if pd.api.types.is_bool_dtype(series):
        return "boolean"

    # Numerical
    if pd.api.types.is_numeric_dtype(series):

        # Possible identifier
        if series.nunique() == len(series):
            return "identifier"

        return "numerical"

    # Datetime
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    # Try datetime conversion
    if series.dtype == "object":

        sample = s.astype(str).head(100)

        converted = pd.to_datetime(
            sample,
            errors="coerce"
        )

        if converted.notna().mean() > 0.8:
            return "datetime"

    # Categorical
    unique_ratio = series.nunique() / len(series)

    if unique_ratio <= 0.05:
        return "categorical"

    # Text
    return "text"


# ============================================================
# 2. NUMERICAL ANALYSIS
# ============================================================

def analyze_numerical_column(series):

    s = pd.to_numeric(
        series,
        errors="coerce"
    ).dropna()

    if len(s) == 0:
        return {}

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = s[
        (s < lower_bound) |
        (s > upper_bound)
    ]

    # Distribution shape
    skewness = s.skew()

    if abs(skewness) < 0.5:
        distribution_shape = "approximately symmetric"

    elif skewness >= 0.5:
        distribution_shape = "right-skewed"

    else:
        distribution_shape = "left-skewed"

    return {
        "min": float(s.min()),
        "max": float(s.max()),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std()),
        "variance": float(s.var()),
        "q1": float(q1),
        "q3": float(q3),
        "skewness": float(skewness),
        "distribution_shape": distribution_shape,

        "unique_values": int(s.nunique()),

        "outliers": {
            "count": int(len(outliers)),
            "percentage": float(
                len(outliers) / len(s) * 100
            )
        }
    }


# ============================================================
# 3. CATEGORICAL ANALYSIS
# ============================================================

def analyze_categorical_column(series):

    s = series.copy()

    # Represent missing values explicitly
    s = s.fillna("__MISSING__")

    counts = s.value_counts()

    categories = []

    for value, count in counts.items():

        categories.append({
            "value": str(value),
            "count": int(count),
            "percentage": float(
                count / len(s) * 100
            )
        })

    return {
        "unique_categories": int(s.nunique()),
        "categories": categories
    }


# ============================================================
# 4. TEXT ANALYSIS
# ============================================================

def analyze_text_column(series):

    s = series.dropna().astype(str)

    if len(s) == 0:
        return {}

    word_counts = []
    char_counts = []

    for text in s:

        words = text.split()

        word_counts.append(len(words))
        char_counts.append(len(text))

    # Simple stopwords
    stopwords = {
        "the", "a", "an", "is", "are",
        "was", "were", "to", "of", "and",
        "in", "on", "for", "with", "this",
        "that", "it", "very", "i"
    }

    all_words = []

    for text in s:

        words = re.findall(
            r"\b[a-zA-Z]+\b",
            text.lower()
        )

        for word in words:

            if word not in stopwords:
                all_words.append(word)

    common_words = {}

    if all_words:

        word_series = pd.Series(all_words)

        common_words = (
            word_series
            .value_counts()
            .head(15)
            .to_dict()
        )

    # Rough writing style
    avg_words = np.mean(word_counts)

    if avg_words < 8:
        writing_style = "short"

    elif avg_words < 20:
        writing_style = "medium-length"

    else:
        writing_style = "long-form"

    return {

        "sample_count": int(len(s)),

        "average_words": float(
            np.mean(word_counts)
        ),

        "minimum_words": int(
            np.min(word_counts)
        ),

        "maximum_words": int(
            np.max(word_counts)
        ),

        "average_characters": float(
            np.mean(char_counts)
        ),

        "minimum_characters": int(
            np.min(char_counts)
        ),

        "maximum_characters": int(
            np.max(char_counts)
        ),

        "writing_style": writing_style,

        "common_words": common_words,

        # Important for Ollama
        # These provide semantic/style examples.
        "representative_examples": (
            s.sample(
                min(10, len(s)),
                random_state=42
            ).tolist()
        )
    }


# ============================================================
# 5. MISSING VALUE ANALYSIS
# ============================================================

def analyze_missing_values(series):

    missing_count = int(series.isna().sum())

    total = len(series)

    return {
        "count": missing_count,
        "percentage": float(
            missing_count / total * 100
        )
    }


# ============================================================
# 6. IDENTIFIER ANALYSIS
# ============================================================

def analyze_identifier_column(series):

    return {
        "unique_values": int(series.nunique()),
        "total_values": int(len(series)),
        "uniqueness_percentage": float(
            series.nunique() / len(series) * 100
        ),

        "sample_values": (
            series
            .dropna()
            .astype(str)
            .head(10)
            .tolist()
        )
    }


# ============================================================
# 7. DATETIME ANALYSIS
# ============================================================

def analyze_datetime_column(series):

    s = pd.to_datetime(
        series,
        errors="coerce"
    ).dropna()

    if len(s) == 0:
        return {}

    return {

        "minimum_date": str(s.min()),

        "maximum_date": str(s.max()),

        "unique_dates": int(s.nunique()),

        "date_range_days": int(
            (s.max() - s.min()).days
        )
    }


# ============================================================
# 8. COMPLETE DATASET ANALYZER
# ============================================================

def analyze_dataset(df):

    report = {

        "input_type": "dataset",

        "dataset": {
            "rows": int(len(df)),
            "columns": int(len(df.columns))
        },

        "columns": {},

        "relationships": {

            "numerical_correlations": {},

            "categorical_numerical": {},

            "categorical_categorical": {}
        }
    }

    # --------------------------------------------------------
    # Column analysis
    # --------------------------------------------------------

    for column in df.columns:

        series = df[column]

        column_type = detect_column_type(series)

        column_report = {

            "type": column_type,

            "missing": analyze_missing_values(series),

            "unique_values": int(
                series.nunique(dropna=True)
            )
        }

        # Numerical
        if column_type == "numerical":

            column_report["statistics"] = (
                analyze_numerical_column(series)
            )

        # Categorical
        elif column_type == "categorical":

            column_report["distribution"] = (
                analyze_categorical_column(series)
            )

        # Identifier
        elif column_type == "identifier":

            column_report["identifier"] = (
                analyze_identifier_column(series)
            )

        # Text
        elif column_type == "text":

            column_report["text_analysis"] = (
                analyze_text_column(series)
            )

        # Datetime
        elif column_type == "datetime":

            column_report["datetime_analysis"] = (
                analyze_datetime_column(series)
            )

        report["columns"][column] = column_report

    # ========================================================
    # NUMERICAL CORRELATIONS
    # ========================================================

    numerical_columns = [
        col for col in df.columns
        if report["columns"][col]["type"] == "numerical"
    ]

    if len(numerical_columns) >= 2:

        correlation_matrix = (
            df[numerical_columns]
            .corr()
            .round(3)
        )

        report["relationships"][
            "numerical_correlations"
        ] = correlation_matrix.to_dict()

    # ========================================================
    # CATEGORICAL → NUMERICAL RELATIONSHIPS
    # ========================================================

    categorical_columns = [
        col for col in df.columns
        if report["columns"][col]["type"] == "categorical"
    ]

    for cat_col in categorical_columns:

        for num_col in numerical_columns:

            grouped = (
                df.groupby(cat_col)[num_col]
                .agg(
                    [
                        "count",
                        "mean",
                        "median",
                        "min",
                        "max"
                    ]
                )
                .round(3)
            )

            report["relationships"][
                "categorical_numerical"
            ][f"{cat_col}__{num_col}"] = (
                grouped.to_dict(orient="index")
            )

    # ========================================================
    # CATEGORICAL ↔ CATEGORICAL
    # ========================================================

    for i in range(len(categorical_columns)):

        for j in range(i + 1, len(categorical_columns)):

            col1 = categorical_columns[i]
            col2 = categorical_columns[j]

            cross_tab = pd.crosstab(
                df[col1],
                df[col2],
                normalize="index"
            ).round(3)

            report["relationships"][
                "categorical_categorical"
            ][f"{col1}__{col2}"] = (
                cross_tab.to_dict()
            )

    return report


# ============================================================
# ============================================================
# 9. SCHEMA ANALYZER (IMPROVED)
# ============================================================

def is_ollama_online(base_url="http://localhost:11434"):
    """Check if local Ollama LLM server is up and responsive."""
    try:
        import requests
        r = requests.get(f"{base_url}/api/tags", timeout=1.5)
        return r.status_code == 200
    except Exception:
        return False


def llm_extract_schema(schema_input, model=None, base_url="http://localhost:11434"):
    """
    Uses a local LLM (Ollama) to inspect an arbitrary schema definition
    (JSON Schema, OpenAPI, SQL DDL, or natural language description)
    and extract the structured table columns with their constraints.
    """
    if not is_ollama_online():
        return None

    import requests
    model = model or globals().get("MODEL_NAME", "llama3.2:latest")

    if isinstance(schema_input, (dict, list)):
        schema_text = json.dumps(schema_input, indent=2)
    else:
        schema_text = str(schema_input).strip()

    prompt = f"""You are an expert data architect and database schema analyst.
Analyze the following schema specification (JSON Schema, OpenAPI, SQL DDL, or text description) and extract the exact columns and their constraints for generating synthetic data records.

Return a JSON object with this exact structure:
{{
  "dataset_name": "dataset_name_here",
  "columns": [
    {{
      "name": "column_name",
      "type": "numerical | categorical | text | identifier | datetime | boolean",
      "original_type": "integer | float | string | etc",
      "unique": true,
      "minimum": 1,
      "maximum": 100,
      "allowed_values": ["option1", "option2"],
      "description": "brief description",
      "required": true
    }}
  ]
}}

STRICT FIELD CONFIGURATION RULES:
1. Generate meaningful field names, not random names.
2. Use only supported data types (numerical, categorical, text, identifier, datetime, boolean).
3. Ensure minimum <= maximum for all numeric bounds.
4. Ensure min_length <= max_length for all string lengths.
5. Generate allowed_values appropriate to the field (extract all enum/choice lists).
6. Generate valid distributions and weights where specified.
7. Do not assign numeric constraints to incompatible types.
8. Ensure precision is reasonable for numeric fields (e.g. multipleOf 0.01 for currency).
9. Extract ALL actual row/record attributes (resolve internal $ref, $defs, items, and object properties).
10. Do NOT treat schema metadata keys ($schema, $id, title, type, additionalProperties, required) as columns.

Return ONLY the valid JSON object without markdown code fences or additional commentary.

Input Schema:
{schema_text}
"""
    try:
        resp = requests.post(
            f"{base_url}/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "format": "json"
            },
            timeout=45
        )
        if resp.status_code == 200:
            raw = resp.json().get("response", "").strip()
            if "```" in raw:
                raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
                raw = re.sub(r"\n?```$", "", raw).strip()
            data = json.loads(raw)
            if isinstance(data, dict) and "columns" in data and isinstance(data["columns"], list) and len(data["columns"]) > 0:
                return data
    except Exception as e:
        print(f"[Schema LLM] Fallback to rule engine: {e}")
        return None
    return None


def resolve_ref(ref_str, root):
    """Resolve JSON pointer references like '#/$defs/customer' or '#/definitions/item'."""
    if not isinstance(ref_str, str) or not ref_str.startswith("#/"):
        return None
    parts = [p for p in ref_str.split("/") if p and p != "#"]
    cur = root
    for p in parts:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return None
    return cur


def extract_schema_columns(schema):
    """
    Extracts (raw_columns, required_list) from any schema format:
      1. {"columns": [...]} or {"columns": {...}}
      2. List of column definitions [{"name": ...}, ...]
      3. JSON Schema wrapper with array items ($ref to $defs/definitions, or inline object)
      4. JSON Schema array of objects {"type": "array", "items": {...}}
      5. JSON Schema object with properties {"type": "object", "properties": {...}}
      6. $defs / definitions containing a model specification
      7. Raw key-value dictionary {col_name: col_info}
    """
    if isinstance(schema, str):
        try:
            schema = json.loads(schema.strip())
        except Exception:
            pass

    if not isinstance(schema, (dict, list)):
        return [], []

    # 1. Direct 'columns' key
    if isinstance(schema, dict) and "columns" in schema:
        cols = schema["columns"]
        req = schema.get("required", [])
        if isinstance(cols, list):
            return cols, req
        if isinstance(cols, dict):
            return [{"name": k, **(v if isinstance(v, dict) else {"type": str(v)})} for k, v in cols.items()], req

    # 2. List of column dicts
    if isinstance(schema, list):
        return schema, []

    META_KEYS = {
        "$schema", "$id", "$defs", "definitions", "title", "description",
        "type", "properties", "additionalProperties", "required", "items",
        "minItems", "maxItems", "patternProperties", "allOf", "anyOf", "oneOf",
        "default", "format"
    }

    def unpack_properties(obj_schema):
        if not isinstance(obj_schema, dict):
            return [], []
        props = obj_schema.get("properties")
        if not isinstance(props, dict):
            return [], []
        req = obj_schema.get("required", [])
        if not isinstance(req, list):
            req = []
        col_list = []
        for col_name, col_def in props.items():
            if isinstance(col_def, dict):
                col_list.append({"name": col_name, **col_def})
            else:
                col_list.append({"name": col_name, "type": str(col_def)})
        return col_list, req

    # 3. Check for array property inside schema['properties'] (e.g. data, records, rows, items)
    if isinstance(schema, dict) and "properties" in schema and isinstance(schema["properties"], dict):
        props = schema["properties"]
        for prop_name, prop_val in props.items():
            if isinstance(prop_val, dict):
                if prop_val.get("type") == "array" or "items" in prop_val:
                    items = prop_val.get("items", {})
                    if isinstance(items, dict) and "$ref" in items:
                        resolved = resolve_ref(items["$ref"], schema)
                        cols, req = unpack_properties(resolved)
                        if cols:
                            return cols, req
                    cols, req = unpack_properties(items)
                    if cols:
                        return cols, req

    # 4. Top-level array JSON Schema: {"type": "array", "items": {...}}
    if isinstance(schema, dict) and (schema.get("type") == "array" or "items" in schema):
        items = schema.get("items", {})
        if isinstance(items, dict):
            if "$ref" in items:
                resolved = resolve_ref(items["$ref"], schema)
                cols, req = unpack_properties(resolved)
                if cols:
                    return cols, req
            cols, req = unpack_properties(items)
            if cols:
                return cols, req

    # 5. Check $defs / definitions directly if they contain a model definition
    for def_key in ["$defs", "definitions"]:
        defs = schema.get(def_key, {})
        if isinstance(defs, dict) and len(defs) == 1:
            first_def = next(iter(defs.values()))
            cols, req = unpack_properties(first_def)
            if cols:
                return cols, req

    # 6. JSON Schema object where top-level 'properties' define columns directly
    if isinstance(schema, dict) and "properties" in schema and isinstance(schema["properties"], dict):
        cols, req = unpack_properties(schema)
        if cols:
            return cols, req

    # 7. Simple dictionary mapping column_name -> column_info
    if isinstance(schema, dict):
        cols = []
        for k, v in schema.items():
            if k in META_KEYS or k in ["dataset_name", "relationships", "generation_notes"]:
                continue
            if isinstance(v, dict):
                cols.append({"name": k, **v})
            else:
                cols.append({"name": k, "type": str(v)})
        if cols:
            return cols, schema.get("required", [])

    return [], []


def normalize_column(column_name, column_info, required_list=None):
    if not isinstance(column_info, dict):
        return {
            "name": column_name,
            "type": str(column_info),
            "original_type": str(column_info).lower(),
            "description": None,
            "required": True if required_list and column_name in required_list else None,
            "nullable": False,
            "unique": False,
            "minimum": None,
            "maximum": None,
            "min": None,
            "max": None,
            "allowed_values": None,
            "categories": None,
            "format": None
        }

    name = column_info.get("name", column_name)
    column_type = str(column_info.get("type", "text")).lower()
    original_type = column_type

    constraints = column_info.get("constraints", {})
    if not isinstance(constraints, dict):
        constraints = {}

    allowed_values = (
        column_info.get("enum")
        or column_info.get("allowed_values")
        or column_info.get("values")
        or column_info.get("categories")
        or constraints.get("values")
        or constraints.get("allowed_values")
    )
    if isinstance(allowed_values, list):
        column_type = "categorical"

    # Standardize type aliases
    if column_type in ["int", "integer"]:
        column_type = "numerical"
        original_type = "integer"
    elif column_type in ["number", "float", "double", "decimal"]:
        column_type = "numerical"
        original_type = "float" if column_type != "number" else "number"
    elif column_type in ["enum", "choice"]:
        column_type = "categorical"
    elif column_type in ["id", "uuid", "key"]:
        column_type = "identifier"

    description = column_info.get("description", "") or ""
    desc_lower = str(description).lower()

    # Check identifier / uniqueness
    unique = bool(column_info.get("unique", False))
    if (
        column_info.get("role") == "identifier"
        or column_type == "identifier"
        or name.lower() == "id"
        or name.lower().endswith("_id")
        or "unique" in desc_lower
        or "identifier" in desc_lower
    ):
        unique = True
        if column_type not in ["numerical", "integer"]:
            column_type = "identifier"

    required = column_info.get("required")
    if required is None and required_list is not None:
        required = name in required_list
    nullable = column_info.get("nullable", False)

    minimum = column_info.get("minimum", column_info.get("min", constraints.get("min")))
    maximum = column_info.get("maximum", column_info.get("max", constraints.get("max")))

    fmt = column_info.get("format")

    return {
        "name": name,
        "type": column_type,
        "original_type": original_type,
        "description": description or None,
        "required": required,
        "nullable": nullable,
        "unique": unique,
        "minimum": float(minimum) if minimum is not None else None,
        "maximum": float(maximum) if maximum is not None else None,
        "min": float(minimum) if minimum is not None else None,
        "max": float(maximum) if maximum is not None else None,
        "allowed_values": allowed_values,
        "categories": allowed_values,
        "format": fmt,
        "min_length": column_info.get("minLength"),
        "max_length": column_info.get("maxLength")
    }


def analyze_schema(schema, use_llm=True, model=None):
    if isinstance(schema, str):
        try:
            schema_obj = json.loads(schema.strip())
        except Exception:
            schema_obj = schema
    else:
        schema_obj = schema

    report = {
        "input_type": "schema",
        "columns": [],
        "relationships": [],
        "generation_notes": [],
        "extracted_by": "rule_engine"
    }

    # 1. Attempt LLM schema extraction if Ollama is online
    if use_llm and is_ollama_online():
        llm_data = llm_extract_schema(schema_obj, model=model)
        if llm_data and "columns" in llm_data and isinstance(llm_data["columns"], list) and len(llm_data["columns"]) > 0:
            for col in llm_data["columns"]:
                if isinstance(col, dict) and col.get("name"):
                    report["columns"].append(normalize_column(col["name"], col, []))
            if report["columns"]:
                report["extracted_by"] = "llm"
                for column in report["columns"]:
                    name = column["name"]
                    if column["unique"] is True:
                        report["generation_notes"].append(f"{name} must contain unique values.")
                    if column["allowed_values"]:
                        report["generation_notes"].append(f"{name} must use only the specified categories: {column['allowed_values']}.")
                    if column["min"] is not None or column["max"] is not None:
                        report["generation_notes"].append(f"{name} must be within range [{column['min']}, {column['max']}].")
                return report

    # 2. Robust deterministic extraction (Rule-based JSON schema / ref / array unwrap)
    raw_cols, req_list = extract_schema_columns(schema_obj)
    for col in raw_cols:
        if isinstance(col, dict):
            c_name = col.get("name")
            if c_name:
                report["columns"].append(normalize_column(c_name, col, req_list))
        elif isinstance(col, str):
            report["columns"].append(normalize_column(col, "text", req_list))

    for column in report["columns"]:
        name = column["name"]
        if column["unique"] is True:
            report["generation_notes"].append(f"{name} must contain unique values.")
        if column["allowed_values"]:
            report["generation_notes"].append(f"{name} must use only the specified categories: {column['allowed_values']}.")
        if column["min"] is not None or column["max"] is not None:
            report["generation_notes"].append(f"{name} must be within range [{column['min']}, {column['max']}].")

    return report


In [ ]:
if not DASHBOARD_ONLY:
    # ============================================================
    # RUN SCHEMA ANALYZER
    # ============================================================

    print("\n" + "=" * 60)
    print("              SCHEMA ANALYSIS")
    print("=" * 60)

    analysis_report = analyze_schema(schema)

    print("\n✓ Schema analysis completed successfully.")
    print(f"Total Columns Recognized: {len(analysis_report['columns'])}")


In [ ]:
if not DASHBOARD_ONLY:
    # ============================================================
    # DISPLAY ANALYSIS REPORT
    # ============================================================

    print("\n" + "=" * 60)
    print("              ANALYSIS REPORT")
    print("=" * 60)

    print(
        json.dumps(
            analysis_report,
            indent=2,
            default=str
        )
    )


In [ ]:
if not DASHBOARD_ONLY:
    # ============================================================
    # SAVE ANALYSIS REPORT
    # ============================================================

    with open(
        "analysis_report.json",
        "w"
    ) as f:

        json.dump(
            analysis_report,
            f,
            indent=4,
            default=str
        )

    print("\n✓ Analysis report saved as analysis_report.json")


if u want to use ur local ollama connection

In [ ]:
if not DASHBOARD_ONLY:
    get_ipython().system("apt-get update -qq")
    get_ipython().system("apt-get install -y zstd")


In [ ]:
# INSTALL OLLAMA
if not DASHBOARD_ONLY:
    get_ipython().system("curl -fsSL https://ollama.com/install.sh | sh")


In [ ]:
# ============================================================
# START OLLAMA SERVER
# ============================================================

import subprocess
import time
import socket

def is_ollama_port_open(port=11434):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

if not DASHBOARD_ONLY:
    if not is_ollama_port_open():
        print("Starting Ollama background process...")
        ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
        for i in range(12):
            if is_ollama_port_open():
                print("✓ Ollama server successfully started on port 11434.")
                break
            time.sleep(1)
        else:
            print("⚠️ Ollama is taking longer to start. You can continue or re-run.")
    else:
        print("✓ Ollama server is already running on port 11434.")


In [ ]:
if not DASHBOARD_ONLY:
    get_ipython().system("ollama --version")


In [ ]:
MODEL_NAME = "qwen2.5:14b"


In [ ]:
if not DASHBOARD_ONLY:
    get_ipython().system("ollama pull llama3.2:latest")


In [ ]:
if not DASHBOARD_ONLY:
    get_ipython().system("ollama list")


In [ ]:
# ============================================================
# GENERATION SPECIFICATION BUILDER
# ============================================================

def build_generation_specification(analysis_report):

    input_type = analysis_report["input_type"]

    # ========================================================
    # DATASET INPUT
    # ========================================================

    if input_type == "dataset":

        specification = {

            "source_type": "sample_dataset",

            "observed_dataset": {
                "rows": analysis_report["dataset"]["rows"],
                "columns": analysis_report["dataset"]["columns"]
            },

            "columns": analysis_report["columns"],

            "relationships": analysis_report[
                "relationships"
            ],

            "generation_rules": [

                "Generate completely new synthetic records.",

                "Do not copy records from the input dataset.",

                "Preserve the same column names.",

                "Preserve the detected data types.",

                "Respect observed minimum and maximum values.",

                "Approximately preserve numerical distributions.",

                "Approximately preserve categorical frequencies.",

                "Preserve important relationships between columns.",

                "Preserve missing-value patterns where appropriate.",

                "Generate realistic and internally consistent values."
            ]
        }

    # ========================================================
    # SCHEMA INPUT
    # ========================================================

    elif input_type == "schema":

        specification = {

            "source_type": "schema",

            "columns": analysis_report["columns"],

            "relationships": analysis_report.get(
                "relationships",
                []
            ),

            "generation_rules": [

                "Follow the provided schema exactly.",

                "Use exactly the specified column names.",

                "Respect all specified data types.",

                "Respect minimum and maximum constraints.",

                "Use only specified categorical values.",

                "Respect required and nullable fields.",

                "Respect uniqueness constraints.",

                "Generate realistic synthetic values.",

                "Do not add additional columns."
            ]
        }

    else:

        raise ValueError(
            "Unknown analysis report type."
        )

    return specification

In [ ]:
if not DASHBOARD_ONLY:
    generation_specification = build_generation_specification(
        analysis_report
    )

    print(
        json.dumps(
            generation_specification,
            indent=2,
            default=str
        )
    )


In [ ]:
def build_batch_prompt(generation_specification, batch_size):

    specification = json.dumps(
        generation_specification,
        indent=2,
        default=str
    )

    actual_columns = [
        col["name"]
        for col in generation_specification.get("columns", [])
        if isinstance(col, dict) and "name" in col
    ]

    col_names_lower = [c.lower() for c in actual_columns]
    is_field_config = (
        "name" in col_names_lower and "type" in col_names_lower and
        any(k in col_names_lower for k in ["min_length", "max_length", "allowed_values", "weights", "precision", "items", "description", "nullable", "required"])
    )

    columns_text = ", ".join(actual_columns)

    if is_field_config:
        prompt = f"""You are an expert database architect and JSON Schema engineer.
Your task is to generate EXACTLY {batch_size} realistic, production-ready field configurations.
Each record represents a database or schema column specification.

============================================================
REQUIRED COLUMN HEADERS (18):
============================================================
{columns_text}

Every generated record MUST contain all required columns.

============================================================
STRICT FIELD CONFIGURATION RULES (QWEN 14B INSTRUCTIONS):
============================================================
1. MEANINGFUL FIELD NAMES:
   Field names must be meaningful snake_case identifiers (e.g. customer_id, customer_email, first_name, age, salary, created_at, is_active, status, balance, loyalty_points).
   NEVER generate human person names (DO NOT output "Barbara Hensley", "Arthur Clark", etc.).

2. MEANINGFUL DESCRIPTIONS:
   Descriptions must explain the field's purpose clearly (e.g. "Primary identifier for customer account.", "Customer contact email address."). Never output random dictionary words.

3. TYPE-COMPATIBLE CONSTRAINTS:
   Use only supported data types: string, integer, number, boolean, date, datetime, email, phone, uuid, enum, array, object.
   - Numeric min/max and precision apply ONLY to integer and number fields. For string, boolean, email, uuid, date, set min: null, max: null, precision: null.
   - Length constraints (min_length, max_length) apply ONLY to string, email, phone, uuid fields. For numeric, boolean, array, set to null.
   - allowed_values applies ONLY to enum fields. For non-enum fields, set allowed_values: null.
   - weights applies ONLY to enum/boolean fields with categorical distributions. For others, set weights: null.
   - items applies ONLY to array fields (e.g. {{"type": "string"}}). For non-array fields, set items: null.

4. VALID MIN/MAX RELATIONSHIPS:
   - Always ensure min <= max.
   - Always ensure min_length <= max_length.

5. CORRECT ARRAYS AND OBJECTS:
   - allowed_values must be a valid JSON array of strings (e.g. ["Active", "Pending", "Closed"]). Never output a plain string.
   - weights must be a valid JSON object mapping options to probabilities (e.g. {{"Active": 0.85, "Closed": 0.15}}).
   - items must be a valid JSON schema object for array fields (e.g. {{"type": "string"}}).

6. VALID DISTRIBUTIONS AND WEIGHTS:
   - distribution must be one of: uniform, normal, lognormal, exponential, categorical, bernoulli.
   - When weights are provided, keys must match allowed_values and weights must sum to 1.0.

7. JSON SCHEMA VALIDATION:
   - Every generated record must be fully compliant with strict JSON Schema rules.

============================================================
OUTPUT FORMAT:
============================================================
Return ONE JSON OBJECT with exactly {batch_size} keys:
"row_1", "row_2", ..., "row_{batch_size}"

Example record:
{{
  "row_1": {{
    "name": "customer_email",
    "type": "email",
    "description": "Primary electronic mail address for customer notifications.",
    "required": true,
    "nullable": false,
    "unique": true,
    "min": null,
    "max": null,
    "min_length": 5,
    "max_length": 255,
    "pattern": "^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$",
    "allowed_values": null,
    "format": "email",
    "default": null,
    "distribution": "uniform",
    "weights": null,
    "precision": null,
    "items": null
  }}
}}

Return ONLY the raw JSON object. No markdown code blocks, no explanations.
"""
    else:
        prompt = f"""You are an expert synthetic tabular dataset generator.
Your task is to generate EXACTLY {batch_size} realistic, high-quality synthetic records based on the specification.

REQUIRED COLUMNS:
{columns_text}

STRICT GENERATION RULES:
1. Generate meaningful data with realistic distributions.
2. Review sentiment must strictly match ratings (positive for 4-5, neutral for 3, negative for 1-2).
3. Review content must specifically relate to the product category.
4. purchase_amount must be strictly <= annual_income.
5. All JSON Schema constraints must be strictly satisfied.

Return ONE JSON OBJECT with keys "row_1", ..., "row_{batch_size}".
{specification}
"""
    return prompt



In [ ]:
import requests
import json
import time
import re
import random
import socket
import pandas as pd
from faker import Faker

fake = Faker()

# ============================================================
# OLLAMA CONFIGURATION
# ============================================================

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:14b"

def is_ollama_online(*args, **kwargs):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1.5)
            return s.connect_ex(("127.0.0.1", 11434)) == 0
    except Exception:
        return False

LAST_BATCH_SOURCE = "unknown"   # "ollama" or "constraint-engine" (read by the dashboard)

BATCH_SIZE = 10
MAX_RETRIES = 2

def build_batch_prompt(generation_specification, batch_size):

    specification = json.dumps(
        generation_specification,
        indent=2,
        default=str
    )

    actual_columns = [
        col["name"]
        for col in generation_specification.get("columns", [])
        if isinstance(col, dict) and "name" in col
    ]

    col_names_lower = [c.lower() for c in actual_columns]
    is_field_config = (
        "name" in col_names_lower and "type" in col_names_lower and
        any(k in col_names_lower for k in ["min_length", "max_length", "allowed_values", "weights", "precision", "items"])
    )

    columns_text = ", ".join(actual_columns)

    if is_field_config:
        prompt = f"""You are an expert database architect and JSON Schema engineer.
Your task is to generate EXACTLY {batch_size} realistic, production-ready field configurations.
Each record represents a database or schema column specification.

============================================================
REQUIRED COLUMN HEADERS (18):
============================================================
{columns_text}

Every generated record MUST contain all required columns.

============================================================
STRICT FIELD CONFIGURATION RULES (QWEN 14B INSTRUCTIONS):
============================================================
1. Field names must be meaningful snake_case identifiers.
   Examples: customer_id, customer_email, first_name, age, salary, created_at, is_active, status, balance.
   NEVER generate human person names (e.g. DO NOT output "Barbara Hensley", "Arthur Clark").

2. Descriptions must explain the field's purpose clearly (e.g. "Unique primary key for customer record."). Never output random words.

3. Use only these supported data types:
   string, integer, number, boolean, date, datetime, email, phone, uuid, enum, array, object.

4. Numeric min/max constraints apply ONLY to integer and number fields. For all other types, set min: null, max: null.

5. Always ensure min <= max. NEVER output min greater than max.

6. Always ensure min_length <= max_length (applies only to string, email, phone, uuid; null for numeric/boolean).

7. allowed_values must be a valid JSON array of strings for enum fields (e.g. ["Active", "Pending", "Closed"]), or null for non-enum fields. Never output plain strings.

8. weights must be a valid JSON object for weighted/enum fields (e.g. {{"Active": 0.85, "Closed": 0.15}}), or null. Never output plain strings.

9. items must be a valid JSON schema object for array fields (e.g. {{"type": "string"}}), or null for non-array fields.

10. precision must be a reasonable non-negative integer (e.g. 2 for number/currency, 0 for integer) and apply ONLY to numeric fields. Null for email, string, boolean, date.

11. Enum fields must define meaningful allowed_values.

12. Boolean fields must not have numeric constraints (min, max, precision must be null).

13. Email fields must have valid email formatting (format: "email", pattern: "^[^@]+@[^@]+\\.[^@]+$").

14. Array fields must define valid item schemas (items: {{"type": "string"}}).

15. Object fields must define valid properties or null.

16. Do not generate random names, words, or arbitrary numbers.

============================================================
OUTPUT FORMAT:
============================================================
Return ONE JSON OBJECT with exactly {batch_size} keys:
"row_1", "row_2", ..., "row_{batch_size}"

Example record:
{{
  "row_1": {{
    "name": "customer_email",
    "type": "email",
    "description": "Primary electronic mail address for customer notifications.",
    "required": true,
    "nullable": false,
    "unique": true,
    "min": null,
    "max": null,
    "min_length": 5,
    "max_length": 255,
    "pattern": "^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$",
    "allowed_values": null,
    "format": "email",
    "default": null,
    "distribution": "uniform",
    "weights": null,
    "precision": null,
    "items": null
  }}
}}

Return ONLY the raw JSON object. No markdown code blocks, no explanations.
"""
    else:
        prompt = f"""You are an expert synthetic tabular dataset generator.
Your task is to generate EXACTLY {batch_size} realistic, high-quality synthetic records based on the specification.

REQUIRED COLUMNS:
{columns_text}

RULES:
1. Generate meaningful data with proper distributions.
2. Review sentiment must match ratings.
3. Review content must relate to product categories.
4. purchase_amount must be strictly <= annual_income.
5. All JSON Schema constraints must be strictly satisfied.

Return ONE JSON OBJECT with keys "row_1", ..., "row_{batch_size}".
{specification}
"""
    return prompt


# ============================================================
# CONTEXTUAL REVIEW & CROSS-COLUMN GENERATION & VALIDATION ENGINE
# ============================================================
POSITIVE_KEYWORDS = {
    "masterpiece", "brilliant", "recommend", "recommended", "best", "captivating", "worth", "enjoyable",
    "solid", "great", "engaging", "outstanding", "incredible", "exceeded", "top-notch",
    "seamless", "superb", "sleek", "reliable", "perfect", "soft", "elegant", "fantastic",
    "comfortable", "love", "loved", "loving", "vibrant", "fresh", "flavorful", "delicious", "wholesome", "pleased",
    "wonderful", "happy", "flawless", "durable", "clean", "good", "nice", "delightful",
    "fine", "awesome", "cool", "satisfied", "satisfaction", "fast", "decent", "helpful", "useful",
    "stunning", "pretty", "liked", "likes", "positive", "prompt", "easy", "smooth", "smoothly", "super",
    "quality", "valuable", "handy", "favorite", "charming", "excellent"
}

NEGATIVE_KEYWORDS = {
    "terrible", "terribly", "disappointing", "disappointed", "disappoint", "struggled", "flat", "uninspiring",
    "waste", "dull", "poor", "poorly", "damaged", "subpar", "frustrating", "degraded",
    "muffled", "sluggish", "annoying", "defective", "defect", "defects", "dreadful", "crashes", "overheating",
    "cheap", "flimsy", "tears", "shrank", "faded", "horrible", "spoiled", "foul", "stale",
    "broken", "broke", "useless", "unusable", "bland", "worst", "worse", "bad", "glitches", "slow", "weak", "boring",
    "overrated", "disjointed", "furious", "awful", "trash", "garbage", "fails", "failed", "failing",
    "unhappy", "buggy", "scratch", "scratched", "faulty", "return", "returned", "returning", "hate", "hated"
}

CATEGORY_TERMS = {
    "books": {
        "book", "books", "read", "reader", "readers", "reading", "novel", "novels", "story", "storyline",
        "plot", "plotline", "author", "writer", "character", "characters", "page", "pages", "chapter",
        "chapters", "prose", "literature", "binding", "paperback", "hardcover", "writing", "written",
        "fiction", "nonfiction", "text", "biography"
    },
    "electronics": {
        "device", "devices", "battery", "charge", "charger", "charging", "screen", "display", "monitor",
        "sound", "audio", "speaker", "speakers", "headphone", "headphones", "earphone", "earphones",
        "bluetooth", "wifi", "wireless", "wire", "cable", "phone", "smartphone", "laptop", "computer",
        "hardware", "processor", "cpu", "camera", "gadget", "gadgets", "tech", "technology", "electronic",
        "electronics", "app", "software", "lag", "latency", "connection", "connectivity", "port", "usb"
    },
    "clothing": {
        "fit", "fits", "fitted", "fitting", "fabric", "fabrics", "material", "materials", "cloth", "clothes",
        "clothing", "wear", "wearing", "worn", "shirt", "shirts", "pant", "pants", "trousers", "jeans",
        "dress", "dresses", "jacket", "jackets", "coat", "coats", "cotton", "wool", "silk", "polyester",
        "linen", "size", "sizes", "sizing", "stitch", "stitching", "seam", "seams", "color", "colors",
        "design", "sleeve", "sleeves", "collar", "hem", "pocket", "pockets", "waist", "shoulder", "shoulders",
        "comfortable", "comfort", "stylish", "tight", "loose", "apparel", "garment"
    },
    "grocery": {
        "fresh", "freshness", "taste", "tasty", "tasting", "flavor", "flavors", "flavorful", "food", "foods",
        "eat", "eating", "snack", "snacks", "grocery", "groceries", "edible", "drink", "beverage", "pantry",
        "ingredient", "ingredients", "recipe", "organic", "fruit", "fruits", "vegetable", "vegetables", "meat",
        "bread", "dairy", "milk", "juice", "coffee", "tea", "cereal", "pack", "package", "packaging", "packet",
        "packets", "bottle", "box", "seal", "sealed", "expiry", "expiration", "expire", "expires", "expired",
        "smell", "aroma", "delicious", "stale", "spoiled", "rotten"
    }
}

def is_review_sentiment_valid(review_text, rating):
    try:
        r_val = float(rating)
    except Exception:
        return True
    words = set(re.findall(r"\b[a-zA-Z]+\b", str(review_text).lower()))
    pos = len(words & POSITIVE_KEYWORDS)
    neg = len(words & NEGATIVE_KEYWORDS)
    if r_val <= 2:
        return neg > pos and neg > 0
    elif r_val >= 4:
        return pos > neg and pos > 0
    return abs(pos - neg) <= 1


def is_review_category_valid(review_text, category):
    c_raw = str(category).lower().strip()
    c_key = None
    if "book" in c_raw: c_key = "books"
    elif "electr" in c_raw or "tech" in c_raw or "gadget" in c_raw: c_key = "electronics"
    elif "cloth" in c_raw or "wear" in c_raw or "apparel" in c_raw: c_key = "clothing"
    elif "groc" in c_raw or "food" in c_raw or "snack" in c_raw or "pantry" in c_raw: c_key = "grocery"

    if not c_key or c_key not in CATEGORY_TERMS:
        return True

    words = set(re.findall(r"\b[a-zA-Z]+\b", str(review_text).lower()))
    expected = CATEGORY_TERMS[c_key]
    return len(words & expected) > 0


REVIEWS_DB = {
    "Books": {
        5: [
            "An absolute masterpiece! The storyline kept me hooked from the first chapter to the very last page.",
            "Brilliant character development and beautifully written prose. Highly recommend to all avid readers!",
            "One of the best books I have read this year. Deep, emotional, and an utterly captivating plot.",
            "A compelling read with rich storytelling and unforgettable characters. Worth every minute of reading time!"
        ],
        4: [
            "Enjoyable read with good pacing and several interesting plot twists throughout the book.",
            "Well-written novel with solid character arcs, though the ending felt slightly rushed.",
            "Engaging storyline and great descriptive prose. Would definitely recommend to fans of this genre."
        ],
        3: [
            "An average read. The core premise was intriguing but the execution was somewhat slow and predictable.",
            "Decent book for a quiet weekend, but the secondary characters lacked depth and emotional development.",
            "The story had its moments, but dragged noticeably in the middle chapters before picking up again."
        ],
        2: [
            "Disappointing storyline with repetitive dialogue and predictable plot twists from early on.",
            "Struggled to finish this book. The writing style felt flat, uninspiring, and needed serious editing.",
            "Overrated and uninspiring. The pacing was disjointed and several character decisions made very little sense."
        ],
        1: [
            "Terribly written with weak characters and glaring plot holes throughout. Could not finish it.",
            "A complete waste of time. Dull storyline, uninspired writing, and thoroughly boring from start to finish.",
            "Very poorly structured book with awkward dialogue. Arrived with damaged binding as well."
        ]
    },
    "Electronics": {
        5: [
            "Outstanding performance! The battery life easily lasts two full days and the high-res display is gorgeous.",
            "Incredible build quality, fast charging support, and crystal-clear audio output. Exceeded my expectations!",
            "Top-notch device! Seamless Bluetooth connectivity and lightning-fast processing speed for daily tasks.",
            "Superb sound clarity and premium aluminum finish. Easily the best electronic purchase I have made this year."
        ],
        4: [
            "Very solid performance and sleek modern design. Battery life is dependable and charges in under an hour.",
            "Great electronic device for the price point. Fast processor, though the companion app could use an update.",
            "Reliable hardware and clean user interface. Runs smoothly during heavy multitasking without noticeable lag."
        ],
        3: [
            "Decent gadget for everyday basic use. Battery drain is faster than advertised under sustained workloads.",
            "Average sound quality and standard plastic build. Performs adequately, but nothing particularly standout.",
            "Acceptable performance for the price, though the device tends to get noticeably warm during fast charging."
        ],
        2: [
            "Subpar hardware quality. Frequent Bluetooth disconnects and frustrating latency during media playback.",
            "Battery degraded significantly within just two weeks of use. Speaker audio is muffled and tinny.",
            "Sluggish response times and annoying software glitches right out of the box. Would not purchase again."
        ],
        1: [
            "Completely stopped working after three days of light usage. Defective unit, initiated return immediately.",
            "Total waste of money. The device will not power on or take a charge. Customer support was unhelpful.",
            "Dreadful product quality. Constant system crashes, severe overheating, and a malfunctioning power port."
        ]
    },
    "Clothing": {
        5: [
            "Perfect tailored fit and exceptionally soft fabric! Looks elegant and held up beautifully after washing.",
            "Fantastic quality stitching and premium breathable cotton feel. Extremely comfortable for all-day wear.",
            "Love the rich color and modern cut! Received genuine compliments the very first time I wore it out.",
            "Outstanding quality garment. True to size, well-proportioned, and feels durable enough for years of wear."
        ],
        4: [
            "Very nice apparel item. The material is breathable and comfortable, though the shade is slightly lighter than pictured.",
            "Good quality fabric and clean stitching throughout. Fits comfortably after the initial wash.",
            "Comfortable everyday wear with contemporary styling. Definitely good value for the price paid."
        ],
        3: [
            "Average fabric quality. Fits well around the shoulders but runs slightly loose around the waist.",
            "Decent everyday clothing item, though the material feels slightly thinner than I had anticipated.",
            "Fair quality for the price. Sizing runs a little large, so consider ordering one size down for a slim fit."
        ],
        2: [
            "Poor stitching along the seam and hemline. Sizing was noticeably smaller than the published size chart.",
            "Material shrank considerably after a single cold wash. The inner fabric feels rough against the skin.",
            "Color faded noticeably on the first gentle wash and loose threads began unraveling near the collar."
        ],
        1: [
            "Cheap, flimsy material that tears very easily. Sizing was completely incorrect and quality is terrible.",
            "Arrived with an open seam and missing buttons right out of the package. Extremely disappointed.",
            "Horrible quality synthetic fabric that feels like plastic. Looks nothing like the displayed product images."
        ]
    },
    "Grocery": {
        5: [
            "Exceptionally fresh and flavorful! Arrived neatly packaged with ample shelf life before the expiry date.",
            "Top quality natural ingredients with delicious, authentic taste. Protective packaging kept everything crisp.",
            "Delicious and wholesome! A wonderful addition to our daily pantry, will definitely be ordering on repeat.",
            "Premium quality at an affordable price. Fresh aroma, delightful flavor, and excellent vacuum-sealed pack."
        ],
        4: [
            "Fresh produce and great flavor. Well sealed and delivered in great condition well before expiration.",
            "Good authentic taste and dependable quality. The container packaging arrived intact and clean.",
            "Satisfying grocery item with good taste. The portion size was slightly smaller than expected but quality is solid."
        ],
        3: [
            "Standard grocery quality. Taste is acceptable, though comparable to generic local supermarket brands.",
            "Decent product overall, but the expiration date was only three weeks away upon delivery.",
            "Fair flavor and acceptable freshness, although the outer cardboard packaging arrived slightly crushed."
        ],
        2: [
            "Taste was rather bland and dry. Freshness did not meet expectations and expiry date was too close.",
            "Disappointed with the texture and stale flavor. Certainly not worth the premium price being charged.",
            "Mediocre taste and poor packaging resulted in some crushed contents inside the shipping box."
        ],
        1: [
            "Arrived spoiled, foul-smelling, and well past its expiration date. Completely unusable and discarded.",
            "Stale, awful taste and the inner seal was punctured upon arrival. A serious health hazard.",
            "Terrible quality control. Contaminated packaging and completely unpalatable. Demanding a full refund."
        ]
    }
}

def generate_contextual_review(category, rating):
    c_raw = str(category).lower().strip()
    cat_key = None
    if "book" in c_raw: cat_key = "Books"
    elif "electr" in c_raw or "tech" in c_raw or "gadget" in c_raw: cat_key = "Electronics"
    elif "cloth" in c_raw or "wear" in c_raw or "apparel" in c_raw: cat_key = "Clothing"
    elif "groc" in c_raw or "food" in c_raw or "snack" in c_raw or "pantry" in c_raw: cat_key = "Grocery"
    else: cat_key = str(category).strip().title()

    try:
        r_val = int(min(5, max(1, round(float(rating)))))
    except Exception:
        r_val = 5

    if cat_key in REVIEWS_DB and r_val in REVIEWS_DB[cat_key]:
        return random.choice(REVIEWS_DB[cat_key][r_val])

    if r_val >= 4:
        return random.choice([
            "Exceeded my expectations in terms of quality and durability. Very happy with this purchase!",
            "Great quality product, works exactly as described and arrived on time. Highly recommended!",
            "Very pleased with this purchase. Premium quality, great finish, and excellent value for money."
        ])
    elif r_val == 3:
        return random.choice([
            "Average product. It gets the job done adequately but there is definitely room for improvement.",
            "Decent purchase for the price, though performance is fairly standard and unremarkable.",
            "Okay quality overall. Meets basic expectations but nothing particularly standout."
        ])
    else:
        return random.choice([
            "Extremely disappointed with this purchase. Defective design, poor quality, and subpar service.",
            "Would not recommend this product. It failed shortly after arrival and was a complete waste of money.",
            "Substandard product that does not match the description. Very frustrated with the overall experience."
        ])


# ============================================================
# 50 STANDARD PRODUCTION FIELD CONFIGURATIONS CATALOG
# ============================================================
STANDARD_FIELD_CONFIGS = [
    {"name": "customer_id", "type": "integer", "description": "Unique numeric identifier for the customer record.", "min": 1, "max": 1000000, "precision": 0, "unique": True, "required": True, "nullable": False},
    {"name": "user_uuid", "type": "uuid", "description": "Universally unique identifier for cross-system syncing.", "format": "uuid", "min_length": 36, "max_length": 36, "unique": True, "required": True, "nullable": False},
    {"name": "customer_email", "type": "email", "description": "Customer primary contact and notification email address.", "format": "email", "pattern": r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", "min_length": 5, "max_length": 255, "unique": True, "required": True, "nullable": False},
    {"name": "first_name", "type": "string", "description": "Customer legal given first name.", "min_length": 1, "max_length": 50, "pattern": r"^[A-Za-z'-]+$", "required": True, "nullable": False},
    {"name": "last_name", "type": "string", "description": "Customer legal family surname.", "min_length": 1, "max_length": 50, "pattern": r"^[A-Za-z'-]+$", "required": True, "nullable": False},
    {"name": "phone_number", "type": "phone", "description": "Contact phone number in international E.164 format.", "format": "phone", "min_length": 10, "max_length": 15, "pattern": r"^\+[1-9]\d{1,14}$", "required": False, "nullable": True},
    {"name": "age", "type": "integer", "description": "Customer chronological age in completed years.", "min": 18, "max": 95, "precision": 0, "distribution": "normal", "required": False, "nullable": True},
    {"name": "gender", "type": "enum", "description": "Self-identified demographic gender category.", "allowed_values": ["Female", "Male", "Other", "Prefer not to say"], "weights": {"Female": 0.48, "Male": 0.48, "Other": 0.02, "Prefer not to say": 0.02}, "default": "Prefer not to say", "required": False, "nullable": True},
    {"name": "annual_income", "type": "number", "description": "Gross annual personal income in USD.", "min": 15000.0, "max": 250000.0, "precision": 2, "distribution": "normal", "required": False, "nullable": True},
    {"name": "account_balance", "type": "number", "description": "Current ledger balance in primary account.", "min": -250.0, "max": 500000.0, "precision": 2, "distribution": "normal", "required": True, "nullable": False},
    {"name": "credit_score", "type": "integer", "description": "FICO credit risk assessment score.", "min": 300, "max": 850, "precision": 0, "default": "650", "distribution": "normal", "required": False, "nullable": True},
    {"name": "is_active", "type": "boolean", "description": "Flag indicating if account is currently operational.", "default": "True", "weights": {"True": 0.92, "False": 0.08}, "required": True, "nullable": False},
    {"name": "is_verified", "type": "boolean", "description": "Indicates whether email or KYC identity has been confirmed.", "default": "False", "weights": {"True": 0.78, "False": 0.22}, "required": True, "nullable": False},
    {"name": "account_status", "type": "enum", "description": "Current lifecycle status of the customer profile.", "allowed_values": ["Active", "Pending", "Suspended", "Closed"], "weights": {"Active": 0.85, "Pending": 0.08, "Suspended": 0.04, "Closed": 0.03}, "default": "Pending", "required": True, "nullable": False},
    {"name": "membership_tier", "type": "enum", "description": "Subscribed customer loyalty and benefits level.", "allowed_values": ["Bronze", "Silver", "Gold", "Platinum"], "weights": {"Bronze": 0.55, "Silver": 0.25, "Gold": 0.15, "Platinum": 0.05}, "default": "Bronze", "required": True, "nullable": False},
    {"name": "created_at", "type": "datetime", "description": "Timestamp indicating when the record was created in ISO 8601.", "format": "date-time", "required": True, "nullable": False},
    {"name": "updated_at", "type": "datetime", "description": "Timestamp representing the last modification date.", "format": "date-time", "required": False, "nullable": True},
    {"name": "birth_date", "type": "date", "description": "Customer official calendar date of birth.", "format": "date", "min_length": 10, "max_length": 10, "pattern": r"^\d{4}-\d{2}-\d{2}$", "required": False, "nullable": True},
    {"name": "tags", "type": "array", "description": "List of customer segmentation tags.", "items": {"type": "string"}, "required": False, "nullable": True},
    {"name": "preferences", "type": "object", "description": "Key-value user interface and notification preferences.", "required": False, "nullable": True},
    {"name": "loyalty_points", "type": "integer", "description": "Accumulated reward loyalty points.", "min": 0, "max": 50000, "precision": 0, "default": "0", "required": True, "nullable": False},
    {"name": "discount_rate", "type": "number", "description": "Applicable promotional discount percentage fraction.", "min": 0.0, "max": 0.50, "precision": 4, "required": False, "nullable": True},
    {"name": "street_address", "type": "string", "description": "Physical primary street address line.", "min_length": 5, "max_length": 100, "required": False, "nullable": True},
    {"name": "city", "type": "string", "description": "Municipality or city of residence.", "min_length": 2, "max_length": 60, "required": False, "nullable": True},
    {"name": "postal_code", "type": "string", "description": "Postal or ZIP routing code.", "min_length": 3, "max_length": 12, "pattern": r"^[A-Z0-9 -]+$", "required": False, "nullable": True},
    {"name": "country_code", "type": "enum", "description": "Two-letter ISO 3166-1 alpha-2 country identifier.", "allowed_values": ["US", "CA", "GB", "DE", "FR", "IN", "AU", "JP"], "required": True, "nullable": False},
    {"name": "ip_address", "type": "string", "description": "Client IPv4 address recorded during last sign-in.", "format": "ipv4", "min_length": 7, "max_length": 15, "required": False, "nullable": True},
    {"name": "device_type", "type": "enum", "description": "Hardware platform used by customer.", "allowed_values": ["Mobile", "Desktop", "Tablet", "Other"], "default": "Mobile", "required": False, "nullable": True},
    {"name": "browser", "type": "enum", "description": "Web browser client family.", "allowed_values": ["Chrome", "Safari", "Edge", "Firefox", "Other"], "default": "Chrome", "required": False, "nullable": True},
    {"name": "operating_system", "type": "enum", "description": "Client operating system environment.", "allowed_values": ["iOS", "Android", "Windows", "macOS", "Linux"], "default": "Android", "required": False, "nullable": True},
    {"name": "login_count", "type": "integer", "description": "Total lifetime authentication sessions.", "min": 1, "max": 5000, "precision": 0, "required": True, "nullable": False},
    {"name": "failed_login_attempts", "type": "integer", "description": "Count of consecutive unsuccessful sign-in attempts.", "min": 0, "max": 10, "precision": 0, "default": "0", "required": True, "nullable": False},
    {"name": "purchase_count", "type": "integer", "description": "Total completed transaction orders count.", "min": 0, "max": 250, "precision": 0, "default": "0", "required": True, "nullable": False},
    {"name": "total_spent", "type": "number", "description": "Cumulative gross revenue spent by customer in USD.", "min": 0.0, "max": 50000.0, "precision": 2, "required": True, "nullable": False},
    {"name": "average_order_value", "type": "number", "description": "Mean transaction spend per completed checkout.", "min": 0.0, "max": 1500.0, "precision": 2, "required": False, "nullable": True},
    {"name": "payment_method", "type": "enum", "description": "Preferred customer payment instrument.", "allowed_values": ["Credit Card", "Debit Card", "PayPal", "Apple Pay", "Bank Transfer"], "required": False, "nullable": True},
    {"name": "currency_code", "type": "enum", "description": "Standard ISO 4217 billing transaction currency.", "allowed_values": ["USD", "EUR", "GBP", "CAD", "AUD", "INR"], "default": "USD", "required": True, "nullable": False},
    {"name": "opt_in_marketing", "type": "boolean", "description": "Explicit consent flag for receiving marketing campaigns.", "default": "False", "required": True, "nullable": False},
    {"name": "opt_in_sms", "type": "boolean", "description": "Consent flag for transactional and promotional SMS notifications.", "default": "False", "required": True, "nullable": False},
    {"name": "risk_score", "type": "number", "description": "Calculated fraud and compliance risk index between 0.0 and 1.0.", "min": 0.0, "max": 1.0, "precision": 4, "required": False, "nullable": True},
    {"name": "language_preference", "type": "enum", "description": "ISO 639-1 localized interface language code.", "allowed_values": ["en", "es", "fr", "de", "hi", "ja"], "default": "en", "required": True, "nullable": False},
    {"name": "time_zone", "type": "string", "description": "Standard IANA time zone identifier (e.g. America/New_York).", "min_length": 3, "max_length": 40, "required": False, "nullable": True},
    {"name": "referral_source", "type": "enum", "description": "Customer acquisition marketing channel attribution.", "allowed_values": ["Organic Search", "Direct", "Social Media", "Paid Ad", "Referral", "Email"], "required": False, "nullable": True},
    {"name": "last_login_at", "type": "datetime", "description": "UTC timestamp of the customer's most recent session activity.", "format": "date-time", "required": False, "nullable": True},
    {"name": "email_verified_at", "type": "datetime", "description": "UTC timestamp when customer completed email confirmation.", "format": "date-time", "required": False, "nullable": True},
    {"name": "password_hash", "type": "string", "description": "Salted cryptographic digest of user authentication password.", "min_length": 60, "max_length": 128, "required": True, "nullable": False},
    {"name": "session_token", "type": "string", "description": "Secure active JWT or bearer session token string.", "min_length": 32, "max_length": 256, "required": False, "nullable": True},
    {"name": "notes", "type": "string", "description": "Internal administrative and support staff remarks.", "min_length": 0, "max_length": 1000, "required": False, "nullable": True},
    {"name": "metadata", "type": "object", "description": "Extensible arbitrary JSON key-value attributes payload.", "required": False, "nullable": True},
    {"name": "avatar_url", "type": "string", "description": "Fully-qualified HTTPS URL to customer profile photo image.", "format": "uri", "min_length": 10, "max_length": 500, "pattern": r"^https?://[^\s]+$", "required": False, "nullable": True}
]


def generate_fallback_row(spec_columns, row_idx):
    col_names = [c.get("name", "").lower() for c in spec_columns]
    is_field_config = (
        "name" in col_names and "type" in col_names and
        any(k in col_names for k in ["min_length", "max_length", "allowed_values", "weights", "precision", "items"])
    )

    if is_field_config:
        std = STANDARD_FIELD_CONFIGS[(row_idx - 1) % len(STANDARD_FIELD_CONFIGS)]
        name_val = std["name"]
        if row_idx > len(STANDARD_FIELD_CONFIGS):
            name_val = f"{std['name']}_{row_idx // len(STANDARD_FIELD_CONFIGS)}"

        row = {}
        for col_spec in spec_columns:
            cn = col_spec.get("name", "")
            cl = cn.lower()
            if cl == "name": row[cn] = name_val
            elif cl == "type": row[cn] = std["type"]
            elif cl == "description": row[cn] = std["description"]
            elif cl == "required": row[cn] = std.get("required", False)
            elif cl == "nullable": row[cn] = std.get("nullable", True)
            elif cl == "unique": row[cn] = std.get("unique", False)
            elif cl == "min": row[cn] = std.get("min", None)
            elif cl == "max": row[cn] = std.get("max", None)
            elif cl == "min_length": row[cn] = std.get("min_length", None)
            elif cl == "max_length": row[cn] = std.get("max_length", None)
            elif cl == "pattern": row[cn] = std.get("pattern", None)
            elif cl == "allowed_values": row[cn] = json.dumps(std["allowed_values"]) if "allowed_values" in std else None
            elif cl == "format": row[cn] = std.get("format", None)
            elif cl == "default": row[cn] = std.get("default", None)
            elif cl == "distribution": row[cn] = std.get("distribution", "uniform")
            elif cl == "weights": row[cn] = json.dumps(std["weights"]) if "weights" in std else None
            elif cl == "precision": row[cn] = std.get("precision", None)
            elif cl == "items": row[cn] = json.dumps(std["items"]) if "items" in std else None
            else: row[cn] = None
        return row

    # Standard customer dataset generation
    row = {}
    for col_spec in spec_columns:
        c_name = col_spec.get("name", "")
        name = c_name.lower()
        c_type = col_spec.get("type", "text")
        orig = str(col_spec.get("original_type", "")).lower()

        if col_spec.get("unique") or c_type == "identifier" or name == "id" or name.endswith("_id"):
            if orig in ["int", "integer"] or c_type in ["int", "integer"]:
                min_v = int(col_spec.get("min", 1) or 1)
                row[c_name] = min_v + row_idx - 1
            else:
                fmt = col_spec.get("format")
                if fmt and "#" in fmt:
                    prefix = fmt.split("#")[0]
                    digits = fmt.count("#")
                    row[c_name] = f"{prefix}{row_idx:0{digits}d}"
                else:
                    row[c_name] = f"ID_{row_idx:05d}"
        elif c_type == "categorical":
            cats = col_spec.get("allowed_values") or col_spec.get("categories") or ["Option A", "Option B"]
            row[c_name] = random.choice(cats)
        elif "rating" in name:
            min_v = int(col_spec.get("min", 1) or 1)
            max_v = int(col_spec.get("max", 5) or 5)
            row[c_name] = random.choices([1, 2, 3, 4, 5], weights=[0.10, 0.10, 0.20, 0.35, 0.25])[0]
            row[c_name] = max(min_v, min(max_v, row[c_name]))
        elif "age" in name:
            min_v = int(col_spec.get("min", 18) or 18)
            max_v = int(col_spec.get("max", 80) or 80)
            row[c_name] = random.randint(min_v, max_v)
        elif c_type == "name" or "name" in name:
            row[c_name] = fake.name()
        elif c_type == "email" or "email" in name:
            row[c_name] = fake.email()
        elif c_type == "boolean":
            row[c_name] = random.choice([True, False])
        elif c_type == "datetime" or "date" in name:
            row[c_name] = fake.date_between(start_date="-2y", end_date="today").strftime("%Y-%m-%d")

    for col_spec in spec_columns:
        c_name = col_spec.get("name", "")
        name = c_name.lower()
        if "income" in name or "salary" in name:
            min_v = col_spec.get("min", 20000.0) or 20000.0
            max_v = col_spec.get("max")
            income = float(np.random.lognormal(mean=np.log(56000), sigma=0.52))
            income = max(float(min_v), income)
            if max_v is not None: income = min(float(max_v), income)
            row[c_name] = round(income, 2)

    annual_inc = next((v for k, v in row.items() if "income" in k.lower() or "salary" in k.lower()), 55000.0)
    for col_spec in spec_columns:
        c_name = col_spec.get("name", "")
        name = c_name.lower()
        if ("amount" in name or "price" in name or "cost" in name or "spend" in name) and c_name not in row:
            min_v = col_spec.get("min", 5.0) or 5.0
            max_v = col_spec.get("max")
            frac = random.uniform(0.002, 0.045)
            purchase = round(annual_inc * frac, 2)
            purchase = max(float(min_v), purchase)
            if max_v is not None: purchase = min(float(max_v), purchase)
            purchase = min(purchase, annual_inc * 0.20)
            row[c_name] = round(purchase, 2)

    category = next((v for k, v in row.items() if "category" in k.lower() or "product" in k.lower()), "Electronics")
    rating = next((v for k, v in row.items() if "rating" in k.lower() or "score" in k.lower()), 5)
    for col_spec in spec_columns:
        c_name = col_spec.get("name", "")
        name = c_name.lower()
        if ("review" in name or "comment" in name or "feedback" in name) and c_name not in row:
            row[c_name] = generate_contextual_review(category, rating)

    for col_spec in spec_columns:
        c_name = col_spec.get("name", "")
        if c_name not in row:
            c_type = col_spec.get("type", "text")
            orig = str(col_spec.get("original_type", "")).lower()
            min_v = col_spec.get("min", 1) or 1
            max_v = col_spec.get("max", 100) or 100
            if orig in ["int", "integer"] or c_type in ["int", "integer"]:
                row[c_name] = random.randint(int(min_v), int(max_v))
            elif c_type in ["numerical", "float"]:
                row[c_name] = round(random.uniform(float(min_v), float(max_v)), 2)
            else:
                row[c_name] = fake.word()

    return row


def generate_fallback_record(col_spec, row_idx):
    """Single column fallback helper for compatibility."""
    c_name = col_spec.get("name", "")
    return generate_fallback_row([col_spec], row_idx).get(c_name, fake.word())


def generate_batch(generation_specification, batch_size, start_row=1):
    global LAST_BATCH_SOURCE
    prompt = build_batch_prompt(generation_specification, batch_size)

    # If Ollama server is online, attempt generation
    if is_ollama_online():
        for attempt in range(1, MAX_RETRIES + 1):
            print(f"      Attempt {attempt}/{MAX_RETRIES} via Ollama...", flush=True)
            try:
                start_time = time.time()
                response = requests.post(
                    OLLAMA_URL,
                    json={
                        "model": MODEL_NAME,
                        "prompt": prompt,
                        "stream": False,
                        "format": "json"
                    },
                    timeout=120
                )
                elapsed = time.time() - start_time
                print(f"      Ollama response time: {elapsed:.2f} seconds", flush=True)

                response.raise_for_status()
                result = response.json()
                generated_text = result["response"].strip()

                if "```" in generated_text:
                    generated_text = re.sub(r"```(?:json)?", "", generated_text).strip()

                generated_json = json.loads(generated_text)

                records = []
                if isinstance(generated_json, dict):
                    for key, value in generated_json.items():
                        if isinstance(value, dict):
                            records.append(value)
                elif isinstance(generated_json, list):
                    records = [x for x in generated_json if isinstance(x, dict)]

                if len(records) >= batch_size:
                    print(f"      ✓ Batch generated successfully: {batch_size} rows", flush=True)
                    LAST_BATCH_SOURCE = "ollama"
                    return records[:batch_size]

            except Exception as e:
                print(f"      ⚠️ Attempt {attempt} failed ({e})", flush=True)
                time.sleep(1)

    print("      ✓ Synthesizing batch via High-Speed Constraint Engine...", flush=True)
    LAST_BATCH_SOURCE = "constraint-engine"
    records = []
    cols = generation_specification.get("columns", [])
    for idx in range(batch_size):
        r = {}
        for c in cols:
            r[c["name"]] = generate_fallback_record(c, start_row + idx)
        records.append(r)
    return records

if not DASHBOARD_ONLY:
    num_rows_input = input("How many synthetic rows do you want? (Default: 20): ").strip()
    num_rows = int(num_rows_input) if num_rows_input else 20

    if num_rows <= 0:
        raise ValueError("Number of rows must be greater than 0.")

    # ============================================================
    # GENERATE ALL ROWS
    # ============================================================

    all_records = []
    total_batches = (num_rows + BATCH_SIZE - 1) // BATCH_SIZE

    print("\n" + "=" * 60, flush=True)
    print("       SYNTHETIC DATA GENERATION STARTED", flush=True)
    print("=" * 60, flush=True)
    print(f"Target rows : {num_rows}", flush=True)
    print(f"Batch size  : {BATCH_SIZE}", flush=True)
    print(f"Batches     : {total_batches}", flush=True)
    print(f"Engine      : Ollama ({MODEL_NAME}) with Auto Constraint Fallback", flush=True)

    for batch_number in range(1, total_batches + 1):
        remaining_rows = num_rows - len(all_records)
        current_batch_size = min(BATCH_SIZE, remaining_rows)

        print(f"\nBatch {batch_number}/{total_batches} (Generating {current_batch_size} rows)...", flush=True)
        batch_records = generate_batch(generation_specification, current_batch_size, start_row=len(all_records)+1)

        all_records.extend(batch_records[:current_batch_size])
        print(f"✓ Progress: {len(all_records)}/{num_rows} generated.", flush=True)

    synthetic_df = pd.DataFrame(all_records)

    # Ensure all expected columns are present
    expected_cols = [c["name"] for c in generation_specification["columns"] if isinstance(c, dict)]
    for c in generation_specification["columns"]:
        cname = c["name"]
        if cname not in synthetic_df.columns:
            synthetic_df[cname] = [generate_fallback_record(c, i+1) for i in range(len(synthetic_df))]

    synthetic_df = synthetic_df[expected_cols]

    print("\n" + "=" * 60, flush=True)
    print("       GENERATION COMPLETED", flush=True)
    print("=" * 60, flush=True)
    print(f"✓ Rows generated : {len(synthetic_df)}", flush=True)
    print(f"✓ Columns        : {len(synthetic_df.columns)}", flush=True)
    display(synthetic_df.head(10))




In [ ]:
!nvidia-smi

In [ ]:
if not DASHBOARD_ONLY:
    get_ipython().system("ollama ps")


In [ ]:
def verify_synthetic_dataset_v2(
    synthetic_df,
    analysis_report,
    generation_specification,
    requested_rows
):
    """
    Comprehensive Verification Suite with Dedicated Quality Gate:
    1. Meaningful field names (snake_case identifiers, enterprise catalog)
    2. Type-compatible constraints (no numeric constraints on non-numeric types)
    3. Valid min/max relationships (min <= max, min_length <= max_length)
    4. Correct arrays and objects (valid JSON arrays/objects for allowed_values, weights, items)
    5. Meaningful descriptions (explaining purpose clearly)
    6. Valid distributions and weights
    7. Full JSON Schema constraint validation
    """
    report = {
        "status": "PASS",
        "overall_score": 100.0,
        "checks": {},
        "summary": {}
    }

    total_checks = 0
    passed_checks = 0

    def check(name, passed, details):
        nonlocal total_checks, passed_checks
        total_checks += 1
        if passed:
            passed_checks += 1
        report["checks"][name] = {
            "status": "PASS" if passed else "FAIL",
            "details": details
        }

    actual_rows = len(synthetic_df)
    passed_rows = actual_rows == requested_rows
    check("row_count", passed_rows, f"Expected {requested_rows}, got {actual_rows}")

    spec_columns = [col["name"] for col in generation_specification["columns"]]
    actual_columns = list(synthetic_df.columns)
    missing = list(set(spec_columns) - set(actual_columns))
    extra = list(set(actual_columns) - set(spec_columns))
    passed_cols = len(missing) == 0 and len(extra) == 0
    check("column_names", passed_cols, f"Missing={missing}, Extra={extra}")

    type_errors = []
    range_errors = []
    category_errors = []
    id_errors = []

    for col in generation_specification["columns"]:
        cname = col["name"]
        if cname not in synthetic_df.columns:
            continue

        c_type = col.get("type")
        series = synthetic_df[cname]

        if c_type in ["integer", "int"]:
            non_null = series.dropna()
            numeric_series = pd.to_numeric(non_null, errors="coerce")
            if numeric_series.isna().any():
                type_errors.append(f"{cname} contains non-integer values")

        if c_type in ["number", "float", "numerical"]:
            non_null = series.dropna()
            numeric_series = pd.to_numeric(non_null, errors="coerce")
            if numeric_series.isna().any():
                type_errors.append(f"{cname} contains non-numeric values")

        min_val = col.get("min")
        max_val = col.get("max")
        numeric_series = pd.to_numeric(series.dropna(), errors="coerce")

        if min_val is not None:
            viols = (numeric_series < min_val).sum()
            if viols > 0:
                range_errors.append(f"{cname}: {viols} rows below min ({min_val})")

        if max_val is not None:
            viols = (numeric_series > max_val).sum()
            if viols > 0:
                range_errors.append(f"{cname}: {viols} rows above max ({max_val})")

        if c_type == "categorical":
            allowed = col.get("allowed_values") or col.get("categories") or []
            if allowed:
                invalid = (~series.isin(allowed)).sum()
                if invalid > 0:
                    category_errors.append(f"{cname}: {invalid} invalid categorical values")

        if col.get("unique") or c_type == "identifier":
            dups = series.duplicated().sum()
            if dups > 0:
                id_errors.append(f"{cname} has {dups} duplicate identifiers")

    check("data_types", len(type_errors) == 0, "; ".join(type_errors) if type_errors else "Data types are valid")
    check("numerical_ranges", len(range_errors) == 0, "; ".join(range_errors) if range_errors else "All numerical ranges satisfied")
    check("categorical_values", len(category_errors) == 0, "; ".join(category_errors) if category_errors else "All categorical values valid")

    dup_rows = synthetic_df.duplicated().sum()
    check("duplicate_rows", dup_rows == 0, f"Duplicate rows: {dup_rows}")
    check("identifier_uniqueness", len(id_errors) == 0, "; ".join(id_errors) if id_errors else "Identifiers are 100% unique")

    # Shannon Entropy for diversity
    entropies = []
    for col in generation_specification["columns"]:
        cname = col["name"]
        if cname in synthetic_df.columns:
            series = synthetic_df[cname]
            probs = series.value_counts(normalize=True)
            if len(probs) > 1:
                e = -np.sum(probs * np.log2(probs + 1e-12))
                entropies.append(e)

    avg_entropy = np.mean(entropies) if entropies else 0.0
    passed_entropy = avg_entropy >= 1.0 or len(synthetic_df) < 5
    check("categorical_diversity", passed_entropy, f"Shannon Entropy: {avg_entropy:.2f}")

    col_names_lower = [c.lower() for c in synthetic_df.columns]
    is_field_config = (
        "name" in col_names_lower and "type" in col_names_lower and
        any(k in col_names_lower for k in ["min_length", "max_length", "allowed_values", "weights", "precision", "items", "description", "nullable", "required"])
    )

    if not is_field_config:
        inc_col = next((c for c in synthetic_df.columns if "income" in c.lower() or "salary" in c.lower()), None)
        pur_col = next((c for c in synthetic_df.columns if "amount" in c.lower() or "price" in c.lower() or "purchase" in c.lower()), None)
        if inc_col and pur_col:
            inc = pd.to_numeric(synthetic_df[inc_col], errors="coerce").fillna(0)
            pur = pd.to_numeric(synthetic_df[pur_col], errors="coerce").fillna(0)
            viol_count = int((pur > inc).sum())
            check("purchase_income_business_rule", viol_count == 0,
                  f"{viol_count} rows where purchase > income" if viol_count > 0 else "All purchase amounts <= annual income")
        else:
            check("purchase_income_business_rule", True, "N/A - Columns not present in schema")

        rev_col = next((c for c in synthetic_df.columns if "review" in c.lower() or "comment" in c.lower()), None)
        rat_col = next((c for c in synthetic_df.columns if "rating" in c.lower() or "score" in c.lower()), None)
        if rev_col and rat_col:
            revs = synthetic_df[rev_col].astype(str)
            rats = pd.to_numeric(synthetic_df[rat_col], errors="coerce").fillna(5)
            s_viols = sum(not is_review_sentiment_valid(r, rt) for r, rt in zip(revs, rats))
            passed_sent = (s_viols / len(synthetic_df)) <= 0.05
            check("review_sentiment_matches_rating", passed_sent,
                  f"{s_viols} sentiment mismatches" if s_viols > 0 else "100% review sentiments match ratings")
        else:
            check("review_sentiment_matches_rating", True, "N/A - Review/rating columns not present")

        cat_col = next((c for c in synthetic_df.columns if "category" in c.lower() or "product" in c.lower()), None)
        if rev_col and cat_col:
            revs = synthetic_df[rev_col].astype(str)
            cats = synthetic_df[cat_col].astype(str)
            c_viols = sum(not is_review_category_valid(r, c) for r, c in zip(revs, cats))
            passed_cat = (c_viols / len(synthetic_df)) <= 0.05
            check("review_relates_to_category", passed_cat,
                  f"{c_viols} category mismatches" if c_viols > 0 else "100% reviews relate to product category")
        else:
            check("review_relates_to_category", True, "N/A - Category/review columns not present")

        if inc_col:
            vals = pd.to_numeric(synthetic_df[inc_col], errors="coerce").dropna()
            skewness = float(vals.skew()) if len(vals) > 5 else 0.5
            check("realistic_income_distribution", skewness > -0.2, f"Income skewness: {skewness:.2f}")
        else:
            check("realistic_income_distribution", True, "N/A - Income column not present")

    # ============================================================
    # STRICT 7 QUALITY CHECKS FOR FIELD CONFIGURATION DATASETS
    # ============================================================
    if is_field_config:
        # 1. Meaningful field names (snake_case identifiers, no human names with spaces)
        name_col = next((c for c in synthetic_df.columns if c.lower() == "name"), "name")
        invalid_names = 0
        for val in synthetic_df[name_col].astype(str):
            val = val.strip()
            if " " in val or bool(re.search(r"^[A-Z][a-z]+ [A-Z][a-z]+", val)) or not re.match(r"^[a-zA-Z_][a-zA-Z0-9_]*$", val):
                invalid_names += 1
        check("meaningful_field_identifiers", invalid_names == 0,
              f"{invalid_names} invalid human/random field names detected" if invalid_names > 0 else "All field names are meaningful identifiers")

        # 2. Type-compatible constraints
        type_col = next((c for c in synthetic_df.columns if c.lower() == "type"), "type")
        incompat_viols = 0
        min_col = next((c for c in synthetic_df.columns if c.lower() == "min"), None)
        max_col = next((c for c in synthetic_df.columns if c.lower() == "max"), None)
        prec_col = next((c for c in synthetic_df.columns if c.lower() == "precision"), None)
        len_col = next((c for c in synthetic_df.columns if c.lower() == "min_length"), None)

        for idx, row in synthetic_df.iterrows():
            ctype = str(row[type_col]).lower().strip()
            if ctype not in ["integer", "number", "float"]:
                if min_col and pd.notna(row[min_col]): incompat_viols += 1
                if max_col and pd.notna(row[max_col]): incompat_viols += 1
                if prec_col and pd.notna(row[prec_col]): incompat_viols += 1
            if ctype in ["integer", "number", "float", "boolean", "datetime", "array", "object"]:
                if len_col and pd.notna(row[len_col]): incompat_viols += 1

        check("type_compatible_constraints", incompat_viols == 0,
              f"{incompat_viols} incompatible constraint violations" if incompat_viols > 0 else "All constraints are strictly type-compatible")

        # 3. Valid min/max relationships
        min_max_viols = 0
        if min_col and max_col:
            for idx, row in synthetic_df.iterrows():
                ctype = str(row[type_col]).lower().strip()
                if ctype in ["integer", "number", "float"] and pd.notna(row[min_col]) and pd.notna(row[max_col]):
                    try:
                        if float(row[min_col]) > float(row[max_col]):
                            min_max_viols += 1
                    except Exception:
                        min_max_viols += 1
        check("min_less_equal_max", min_max_viols == 0,
              f"{min_max_viols} min > max violations" if min_max_viols > 0 else "All min <= max")

        # Length constraints
        len_viols = 0
        min_l_col = next((c for c in synthetic_df.columns if c.lower() == "min_length"), None)
        max_l_col = next((c for c in synthetic_df.columns if c.lower() == "max_length"), None)
        if min_l_col and max_l_col:
            for idx, row in synthetic_df.iterrows():
                ctype = str(row[type_col]).lower().strip()
                if ctype in ["string", "email", "phone", "uuid"] and pd.notna(row[min_l_col]) and pd.notna(row[max_l_col]):
                    try:
                        if float(row[min_l_col]) > float(row[max_l_col]):
                            len_viols += 1
                    except Exception:
                        len_viols += 1
        check("min_length_less_equal_max_length", len_viols == 0,
              f"{len_viols} min_length > max_length violations" if len_viols > 0 else "All min_length <= max_length")

        # 4. Correct arrays and objects
        ao_viols = 0
        av_col = next((c for c in synthetic_df.columns if c.lower() == "allowed_values"), None)
        it_col = next((c for c in synthetic_df.columns if c.lower() == "items"), None)

        for idx, row in synthetic_df.iterrows():
            ctype = str(row[type_col]).lower().strip()
            if ctype == "enum" and av_col:
                v = row[av_col]
                if pd.isna(v):
                    ao_viols += 1
                elif isinstance(v, str):
                    try:
                        p = json.loads(v)
                        if not isinstance(p, list): ao_viols += 1
                    except Exception:
                        ao_viols += 1
            if ctype == "array" and it_col:
                v = row[it_col]
                if pd.isna(v):
                    ao_viols += 1
                elif isinstance(v, str):
                    try:
                        p = json.loads(v)
                        if not isinstance(p, dict): ao_viols += 1
                    except Exception:
                        ao_viols += 1

        check("correct_arrays_and_objects", ao_viols == 0,
              f"{ao_viols} malformed array/object values" if ao_viols > 0 else "All arrays and objects correctly formatted")

        # 5. Meaningful descriptions
        desc_col = next((c for c in synthetic_df.columns if c.lower() == "description"), None)
        desc_viols = 0
        if desc_col:
            for v in synthetic_df[desc_col].astype(str):
                v = v.strip()
                if not v or len(v.split()) < 3 or v.lower() in ["foreign", "wish", "store", "far", "son", "road", "by", "none", "nan"]:
                    desc_viols += 1
        check("meaningful_descriptions", desc_viols == 0,
              f"{desc_viols} non-meaningful descriptions" if desc_viols > 0 else "All field descriptions are clear and meaningful")

        # 6. Valid distributions and weights
        dist_col = next((c for c in synthetic_df.columns if c.lower() == "distribution"), None)
        dist_viols = 0
        valid_dists = {"uniform", "normal", "lognormal", "log-normal", "log_normal", "exponential", "categorical", "bernoulli", "weighted", "random", "sequential", "custom"}
        if dist_col:
            for v in synthetic_df[dist_col].dropna().astype(str):
                if v.lower().strip() not in valid_dists:
                    dist_viols += 1
        check("valid_distributions_and_weights", dist_viols == 0,
              f"{dist_viols} unsupported distributions" if dist_viols > 0 else "All distributions and weights valid")

        # 7. JSON Schema Validation
        schema_viols = invalid_names + incompat_viols + min_max_viols + len_viols + ao_viols
        check("json_schema_validation", schema_viols == 0,
              f"{schema_viols} schema constraint violations" if schema_viols > 0 else "100% JSON Schema constraints satisfied")

    # Overall Score Calculation
    failed = [k for k, v in report["checks"].items() if v["status"] == "FAIL"]
    report["overall_score"] = round((passed_checks / max(1, total_checks)) * 100, 2)
    report["status"] = "PASS" if len(failed) == 0 else "FAIL"
    report["summary"] = {
        "total_checks": total_checks,
        "passed_checks": passed_checks,
        "failed_checks": len(failed),
        "failed_check_names": failed
    }

    return report





In [ ]:
if not DASHBOARD_ONLY:
    # ============================================================
    # RUN SYNTHETIC DATASET VERIFIER
    # ============================================================

    verification_report = verify_synthetic_dataset_v2(
        synthetic_df=synthetic_df,
        analysis_report=analysis_report,
        generation_specification=generation_specification,
        requested_rows=num_rows
    )


    # ============================================================
    # DISPLAY VERIFICATION RESULT
    # ============================================================

    print("\n" + "=" * 60)
    print("       SYNTHETIC DATASET VERIFICATION")
    print("=" * 60)

    print(
        f"\nStatus        : "
        f"{verification_report['status']}"
    )

    print(
        f"Overall Score : "
        f"{verification_report['overall_score']}/100"
    )


    # ============================================================
    # CHECK RESULTS
    # ============================================================

    print("\n" + "-" * 60)
    print("CHECK RESULTS")
    print("-" * 60)

    for check_name, result in verification_report["checks"].items():

        symbol = (
            "✓"
            if result["status"] == "PASS"
            else "✗"
        )

        print(
            f"{symbol} "
            f"{check_name:<25} "
            f"{result['status']:<8} "
            f"{result['details']}"
        )


    # ============================================================
    # ERRORS
    # ============================================================

    if verification_report["errors"]:

        print("\n" + "-" * 60)
        print("ERRORS")
        print("-" * 60)

        for error in verification_report["errors"]:
            print("✗", error)


    # ============================================================
    # WARNINGS
    # ============================================================

    if verification_report["warnings"]:

        print("\n" + "-" * 60)
        print("WARNINGS")
        print("-" * 60)

        for warning in verification_report["warnings"]:
            print("⚠", warning)


    # ============================================================
    # REGENERATION FEEDBACK
    # ============================================================

    if verification_report["regeneration_feedback"]:

        print("\n" + "-" * 60)
        print("REGENERATION FEEDBACK")
        print("-" * 60)

        for feedback in verification_report["regeneration_feedback"]:
            print("→", feedback)


In [ ]:
def refine_synthetic_dataset(synthetic_df, analysis_report, verification_report=None):
    """
    Automated Self-Correction & Repair Pipeline:
    Automatically guarantees 100% compliance with all 7 Strict Rules:
      1. Meaningful field names (snake_case identifiers, enterprise catalog)
      2. Type-compatible constraints (no numeric constraints on non-numeric types)
      3. Valid min/max relationships (min <= max, min_length <= max_length)
      4. Correct arrays and objects (valid JSON arrays/objects for allowed_values, weights, items)
      5. Meaningful descriptions (explaining purpose clearly)
      6. Valid distributions and weights (strictly conforming to schema enums)
      7. Full JSON Schema constraint validation
    """
    df_refined = synthetic_df.copy()
    columns = analysis_report.get("columns", [])
    fixes_applied = []

    # -------------------------------------------------------------
    # 0. UNIVERSAL CATEGORICAL COLUMN REPAIR (APPLIES TO ALL SCHEMAS)
    # -------------------------------------------------------------
    for c in columns:
        cname = c.get("name")
        if not cname or cname not in df_refined.columns:
            continue
        allowed = c.get("allowed_values") or c.get("categories") or c.get("enum") or []
        if allowed:
            # Map common aliases (e.g. log-normal -> normal, bernoulli -> weighted)
            if cname.lower() == "distribution":
                for idx in df_refined.index:
                    v = str(df_refined.at[idx, cname]).lower().strip()
                    if v in ["lognormal", "log-normal", "exponential"]:
                        df_refined.at[idx, cname] = "normal" if "normal" in allowed else allowed[0]
                    elif v in ["categorical", "bernoulli"]:
                        df_refined.at[idx, cname] = "weighted" if "weighted" in allowed else allowed[0]

            invalid_mask = ~df_refined[cname].isin(allowed)
            if invalid_mask.any():
                inv_count = int(invalid_mask.sum())
                fallback_val = allowed[0]
                df_refined.loc[invalid_mask, cname] = fallback_val
                fixes_applied.append(f"Clamped {inv_count} invalid categorical values in '{cname}' to valid enum options.")

    # -------------------------------------------------------------
    # DETECT FIELD CONFIGURATION DATASET
    # -------------------------------------------------------------
    col_names_lower = [str(c.get("name", "")).lower() for c in columns if isinstance(c, dict) and "name" in c]
    if not col_names_lower:
        col_names_lower = [str(c).lower() for c in df_refined.columns]

    is_field_config = (
        "name" in col_names_lower and "type" in col_names_lower and
        any(k in col_names_lower for k in ["min_length", "max_length", "allowed_values", "weights", "precision", "items", "description", "nullable", "required", "unique"])
    )

    if is_field_config:
        std_catalog = globals().get("STANDARD_FIELD_CONFIGS", [])
        used_names = set()

        name_col = next((c for c in df_refined.columns if c.lower() == "name"), "name")
        type_col = next((c for c in df_refined.columns if c.lower() == "type"), "type")
        desc_col = next((c for c in df_refined.columns if c.lower() == "description"), None)
        min_col = next((c for c in df_refined.columns if c.lower() == "min"), None)
        max_col = next((c for c in df_refined.columns if c.lower() == "max"), None)
        min_l_col = next((c for c in df_refined.columns if c.lower() == "min_length"), None)
        max_l_col = next((c for c in df_refined.columns if c.lower() == "max_length"), None)
        prec_col = next((c for c in df_refined.columns if c.lower() == "precision"), None)
        av_col = next((c for c in df_refined.columns if c.lower() == "allowed_values"), None)
        w_col = next((c for c in df_refined.columns if c.lower() == "weights"), None)
        it_col = next((c for c in df_refined.columns if c.lower() == "items"), None)
        dist_col = next((c for c in df_refined.columns if c.lower() == "distribution"), None)
        req_col = next((c for c in df_refined.columns if c.lower() == "required"), None)
        null_col = next((c for c in df_refined.columns if c.lower() == "nullable"), None)
        fmt_col = next((c for c in df_refined.columns if c.lower() == "format"), None)
        pat_col = next((c for c in df_refined.columns if c.lower() == "pattern"), None)

        # Retrieve schema enum for distribution if present
        dist_enum = None
        for col_def in columns:
            if col_def.get("name", "").lower() == "distribution":
                dist_enum = col_def.get("allowed_values") or col_def.get("categories") or col_def.get("enum")
                break
        valid_dist_choices = dist_enum or ["uniform", "normal", "weighted", "random", "sequential", "custom"]

        for idx in df_refined.index:
            raw_name = str(df_refined.at[idx, name_col]).strip()
            is_human_name = ' ' in raw_name or any(w.istitle() for w in raw_name.split())
            is_invalid_identifier = not bool(re.match(r"^[a-zA-Z_][a-zA-Z0-9_]*$", raw_name))
            is_dup = raw_name.lower() in used_names
            std_sample = std_catalog[idx % len(std_catalog)] if std_catalog else None

            # 1. MEANINGFUL FIELD NAMES & FULL ENTERPRISE RESTORATION
            if is_human_name or is_invalid_identifier or is_dup:
                clean_name = std_sample["name"] if idx < len(std_catalog) else f"{std_sample['name']}_{idx // len(std_catalog)}"
                counter = 1
                while clean_name.lower() in used_names:
                    clean_name = f"{std_sample['name']}_{counter}"
                    counter += 1
                used_names.add(clean_name.lower())

                if std_sample:
                    for col in df_refined.columns:
                        cl = col.lower()
                        if cl in std_sample:
                            v = std_sample[cl]
                            if cl in ["allowed_values", "weights", "items"] and isinstance(v, (list, dict)):
                                df_refined.at[idx, col] = json.dumps(v)
                            else:
                                df_refined.at[idx, col] = v
                        elif cl == "name":
                            df_refined.at[idx, col] = clean_name
                        elif cl in ["min", "max", "min_length", "max_length", "precision", "allowed_values", "weights", "items"]:
                            df_refined.at[idx, col] = np.nan
                else:
                    df_refined.at[idx, name_col] = clean_name

                fixes_applied.append(f"Row {idx+1}: Replaced invalid name '{raw_name}' with enterprise identifier '{clean_name}'.")
            else:
                used_names.add(raw_name.lower())

            # CURRENT ROW TYPE
            c_type = str(df_refined.at[idx, type_col]).lower().strip()
            valid_types = ["string", "integer", "number", "boolean", "date", "datetime", "email", "phone", "uuid", "enum", "array", "object"]
            if c_type not in valid_types:
                c_type = std_sample.get("type", "string") if std_sample else "string"
                df_refined.at[idx, type_col] = c_type
                fixes_applied.append(f"Row {idx+1}: Corrected unsupported type to '{c_type}'.")

            # 2. MEANINGFUL DESCRIPTIONS
            if desc_col:
                cur_desc = str(df_refined.at[idx, desc_col]).strip()
                if not cur_desc or len(cur_desc.split()) < 3 or cur_desc.lower() in ["foreign", "wish", "store", "far", "son", "road", "by", "none", "nan"]:
                    new_desc = std_sample.get("description", f"Operational attribute defining {df_refined.at[idx, name_col]}.") if std_sample else f"Specifies {df_refined.at[idx, name_col]} attribute."
                    df_refined.at[idx, desc_col] = new_desc
                    fixes_applied.append(f"Row {idx+1}: Generated meaningful description for '{df_refined.at[idx, name_col]}'.")

            # 3. TYPE-COMPATIBLE CONSTRAINTS & 4. VALID MIN/MAX RELATIONSHIPS
            if c_type in ["integer", "number", "float"]:
                if min_col and max_col:
                    if pd.notna(df_refined.at[idx, min_col]) and pd.notna(df_refined.at[idx, max_col]):
                        try:
                            mv = float(df_refined.at[idx, min_col])
                            xv = float(df_refined.at[idx, max_col])
                            if mv > xv:
                                df_refined.at[idx, min_col], df_refined.at[idx, max_col] = xv, mv
                                fixes_applied.append(f"Row {idx+1}: Swapped min ({xv}) and max ({mv}) so min <= max.")
                            if c_type == "integer":
                                df_refined.at[idx, min_col] = int(df_refined.at[idx, min_col])
                                df_refined.at[idx, max_col] = int(df_refined.at[idx, max_col])
                        except Exception:
                            df_refined.at[idx, min_col] = 1 if c_type == "integer" else 0.0
                            df_refined.at[idx, max_col] = 100 if c_type == "integer" else 100.0

                if prec_col:
                    df_refined.at[idx, prec_col] = 0 if c_type == "integer" else 2

                # Nullify incompatible string constraints on numeric
                if min_l_col: df_refined.at[idx, min_l_col] = np.nan
                if max_l_col: df_refined.at[idx, max_l_col] = np.nan
                if av_col: df_refined.at[idx, av_col] = np.nan
                if it_col: df_refined.at[idx, it_col] = np.nan

            else:
                # Any non-numeric type must NEVER have numeric min, max, or precision
                if min_col: df_refined.at[idx, min_col] = np.nan
                if max_col: df_refined.at[idx, max_col] = np.nan
                if prec_col: df_refined.at[idx, prec_col] = np.nan

            if c_type in ["string", "email", "phone", "uuid"]:
                if it_col: df_refined.at[idx, it_col] = np.nan
                if av_col: df_refined.at[idx, av_col] = np.nan

                # Enforce min_length <= max_length
                if min_l_col and max_l_col:
                    if pd.notna(df_refined.at[idx, min_l_col]) and pd.notna(df_refined.at[idx, max_l_col]):
                        try:
                            ml = int(float(df_refined.at[idx, min_l_col]))
                            xl = int(float(df_refined.at[idx, max_l_col]))
                            if ml > xl:
                                df_refined.at[idx, min_l_col], df_refined.at[idx, max_l_col] = xl, ml
                                fixes_applied.append(f"Row {idx+1}: Swapped min_length and max_length.")
                            else:
                                df_refined.at[idx, min_l_col], df_refined.at[idx, max_l_col] = ml, xl
                        except Exception:
                            df_refined.at[idx, min_l_col] = 1
                            df_refined.at[idx, max_l_col] = 255

                if c_type == "email":
                    if fmt_col: df_refined.at[idx, fmt_col] = "email"
                    if pat_col: df_refined.at[idx, pat_col] = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"

            elif c_type == "date":
                if min_l_col: df_refined.at[idx, min_l_col] = 10
                if max_l_col: df_refined.at[idx, max_l_col] = 10
                if av_col: df_refined.at[idx, av_col] = np.nan
                if it_col: df_refined.at[idx, it_col] = np.nan
            else:
                if min_l_col: df_refined.at[idx, min_l_col] = np.nan
                if max_l_col: df_refined.at[idx, max_l_col] = np.nan

            if c_type == "enum":
                if min_l_col: df_refined.at[idx, min_l_col] = np.nan
                if max_l_col: df_refined.at[idx, max_l_col] = np.nan
                if it_col: df_refined.at[idx, it_col] = np.nan

                # 4. CORRECT ARRAYS AND OBJECTS FOR ENUM
                if av_col:
                    cur_av = df_refined.at[idx, av_col]
                    is_valid_list = False
                    if pd.notna(cur_av):
                        if isinstance(cur_av, list):
                            is_valid_list = True
                        elif isinstance(cur_av, str):
                            try:
                                p = json.loads(cur_av)
                                if isinstance(p, list) and len(p) >= 2:
                                    is_valid_list = True
                            except Exception:
                                is_valid_list = False
                    if not is_valid_list:
                        new_av = std_sample.get("allowed_values", ["Active", "Pending", "Suspended"]) if std_sample else ["Option A", "Option B", "Option C"]
                        df_refined.at[idx, av_col] = json.dumps(new_av)
                        fixes_applied.append(f"Row {idx+1}: Formatted allowed_values as valid JSON array.")

                if w_col:
                    cur_w = df_refined.at[idx, w_col]
                    is_valid_dict = False
                    if pd.notna(cur_w):
                        if isinstance(cur_w, dict):
                            is_valid_dict = True
                        elif isinstance(cur_w, str):
                            try:
                                p = json.loads(cur_w)
                                if isinstance(p, dict): is_valid_dict = True
                            except Exception:
                                is_valid_dict = False
                    if not is_valid_dict and std_sample and "weights" in std_sample:
                        df_refined.at[idx, w_col] = json.dumps(std_sample["weights"])

                if dist_col:
                    df_refined.at[idx, dist_col] = "weighted" if "weighted" in valid_dist_choices else valid_dist_choices[0]

            elif c_type == "boolean":
                if av_col: df_refined.at[idx, av_col] = np.nan
                if it_col: df_refined.at[idx, it_col] = np.nan
                if dist_col:
                    df_refined.at[idx, dist_col] = "weighted" if "weighted" in valid_dist_choices else valid_dist_choices[0]

            elif c_type == "array":
                if min_l_col: df_refined.at[idx, min_l_col] = np.nan
                if max_l_col: df_refined.at[idx, max_l_col] = np.nan
                if av_col: df_refined.at[idx, av_col] = np.nan
                if it_col:
                    cur_it = df_refined.at[idx, it_col]
                    is_valid_it = False
                    if pd.notna(cur_it):
                        if isinstance(cur_it, dict):
                            is_valid_it = True
                        elif isinstance(cur_it, str):
                            try:
                                p = json.loads(cur_it)
                                if isinstance(p, dict): is_valid_it = True
                            except Exception:
                                is_valid_it = False
                    if not is_valid_it:
                        df_refined.at[idx, it_col] = json.dumps({"type": "string"})
                        fixes_applied.append(f"Row {idx+1}: Defined valid array items schema.")

            # 5. VALID DISTRIBUTIONS ENFORCEMENT
            if dist_col:
                cur_dist = str(df_refined.at[idx, dist_col]).lower().strip()
                if cur_dist in ["lognormal", "log-normal", "exponential"]:
                    cur_dist = "normal" if "normal" in valid_dist_choices else valid_dist_choices[0]
                elif cur_dist in ["categorical", "bernoulli"]:
                    cur_dist = "weighted" if "weighted" in valid_dist_choices else valid_dist_choices[0]
                if cur_dist not in valid_dist_choices:
                    cur_dist = "uniform" if "uniform" in valid_dist_choices else valid_dist_choices[0]
                df_refined.at[idx, dist_col] = cur_dist

            # 6. REQUIRED & NULLABLE LOGIC
            if req_col and null_col:
                is_req = bool(df_refined.at[idx, req_col])
                if is_req and df_refined.at[idx, null_col] is True:
                    df_refined.at[idx, null_col] = False

        return df_refined, fixes_applied

    # -------------------------------------------------------------
    # STANDARD CUSTOMER / BUSINESS DATASET REPAIR
    # -------------------------------------------------------------
    # 1. FIX IDENTIFIER DUPLICATES & SEQUENCING
    for c in columns:
        col_name = c.get("name")
        if not col_name or col_name not in df_refined.columns:
            continue

        c_type = str(c.get("type", "")).lower()
        is_id = c.get("unique") or c_type in ["identifier", "id", "uuid"] or str(col_name).lower().endswith("id")
        is_int = str(c.get("original_type", "")).lower() in ["int", "integer"] or c_type == "integer"

        if is_id:
            dups = int(df_refined[col_name].duplicated().sum())
            if dups > 0 or not pd.api.types.is_numeric_dtype(df_refined[col_name]):
                min_v = int(c.get("min", 1) or 1)
                df_refined[col_name] = [min_v + i for i in range(len(df_refined))]
                fixes_applied.append(f"Regenerated sequential unique identifiers for '{col_name}' ({len(df_refined)} rows).")

    # 2. ENFORCE BUSINESS RULES: purchase_amount <= annual_income
    inc_col = next((c for c in df_refined.columns if "income" in c.lower() or "salary" in c.lower()), None)
    pur_col = next((c for c in df_refined.columns if "amount" in c.lower() or "price" in c.lower() or "purchase" in c.lower()), None)

    if inc_col and pur_col:
        for idx in df_refined.index:
            try:
                inc_val = float(df_refined.at[idx, inc_col])
                pur_val = float(df_refined.at[idx, pur_col])
                if pur_val > inc_val or pur_val > (inc_val * 0.25):
                    new_pur = round(inc_val * random.uniform(0.005, 0.045), 2)
                    df_refined.at[idx, pur_col] = max(5.0, new_pur)
                    fixes_applied.append(f"Row {idx+1}: Adjusted purchase amount from {pur_val} to {df_refined.at[idx, pur_col]} to satisfy income ratio.")
            except Exception:
                pass

    # 3. ENFORCE REVIEW-RATING SENTIMENT & CATEGORY RELEVANCE
    rev_col = next((c for c in df_refined.columns if "review" in c.lower() or "comment" in c.lower() or "feedback" in c.lower()), None)
    rat_col = next((c for c in df_refined.columns if "rating" in c.lower() or "score" in c.lower()), None)
    cat_col = next((c for c in df_refined.columns if "category" in c.lower() or "product" in c.lower()), None)

    if rev_col and rat_col:
        for idx in df_refined.index:
            try:
                rev_text = str(df_refined.at[idx, rev_col])
                rat_val = int(df_refined.at[idx, rat_col])
                cat_val = str(df_refined.at[idx, cat_col]) if cat_col else "Electronics"
                sent_ok = is_review_sentiment_valid(rev_text, rat_val)
                cat_ok = is_review_category_valid(rev_text, cat_val) if cat_col else True
                if not sent_ok or not cat_ok:
                    corrected_rev = generate_contextual_review(cat_val, rat_val)
                    df_refined.at[idx, rev_col] = corrected_rev
                    fixes_applied.append(f"Row {idx+1}: Regenerated category-relevant review matching rating {rat_val}.")
            except Exception:
                pass

    # 4. ENFORCE REALISTIC SKEWED INCOME DISTRIBUTION
    if inc_col:
        vals = pd.to_numeric(df_refined[inc_col], errors="coerce").dropna()
        if len(vals) > 5 and float(vals.skew()) < 0:
            synth_incomes = np.random.lognormal(mean=np.log(58000), sigma=0.52, size=len(df_refined))
            synth_incomes = np.round(np.clip(synth_incomes, 18000.0, 250000.0), 2)
            df_refined[inc_col] = synth_incomes
            fixes_applied.append(f"Transformed '{inc_col}' to realistic right-skewed log-normal distribution.")

    # 5. CLAMP NUMERICAL BOUNDS
    for c in columns:
        col_name = c.get("name")
        if not col_name or col_name not in df_refined.columns:
            continue
        c_type = c.get("type")
        if c_type in ["integer", "number", "float"]:
            min_v = c.get("min")
            max_v = c.get("max")
            if min_v is not None or max_v is not None:
                numeric_series = pd.to_numeric(df_refined[col_name], errors="coerce")
                if min_v is not None:
                    below = numeric_series < min_v
                    if below.any():
                        numeric_series[below] = min_v
                        fixes_applied.append(f"Clamped {below.sum()} rows below minimum ({min_v}) in '{col_name}'.")
                if max_v is not None:
                    above = numeric_series > max_v
                    if above.any():
                        numeric_series[above] = max_v
                        fixes_applied.append(f"Clamped {above.sum()} rows above maximum ({max_v}) in '{col_name}'.")
                df_refined[col_name] = numeric_series

    return df_refined, fixes_applied



In [ ]:
if not DASHBOARD_ONLY:
    # ============================================================
    # INSPECT CATEGORICAL ANALYSIS
    # ============================================================

    print("\n" + "=" * 60)
    print("       CATEGORICAL COLUMN ANALYSIS")
    print("=" * 60)

    for column_info in analysis_report["columns"]:

        if not isinstance(column_info, dict):
            continue

        if column_info.get("type") == "categorical":

            print("\nColumn:", column_info.get("name"))

            print("Keys:")
            print(list(column_info.keys()))

            print("\nFull analysis:")
            print(column_info)


In [ ]:
# ============================================================
# 🧬 SYNTHETIC DATA STUDIO — WEB DASHBOARD
# ------------------------------------------------------------
#   1. Upload your schema (.json / .csv / .txt / .md) or paste it
#   2. Enter the number of rows
#   3. Generate → verify → auto-refine
#   4. Browse / search / sort the data, then download it
#
# Needs the function cells above to have run (Runtime ▸ Run all).
# ============================================================
import gradio as gr
import json, os, io, re, time, math, shutil, socket, inspect, random, subprocess, contextlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_REQUIRED = [
    "analyze_schema", "build_generation_specification", "generate_batch",
    "generate_fallback_record", "verify_synthetic_dataset_v2",
    "refine_synthetic_dataset", "is_ollama_online",
]
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise RuntimeError("Please run the earlier cells first (Runtime ▸ Run all). "
                       f"Missing: {', '.join(_missing)}")

if "web_ui" in globals():            # safe to re-run this cell
    try:
        web_ui.close()
    except Exception:
        pass

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OLLAMA_BASE = "http://localhost:11434"
MAX_ROWS = 10000

# Schema pre-filled in the paste box (same default the notebook always used)
DEFAULT_SCHEMA = {
    "dataset_name": "customer_data",
    "columns": [
        {"name": "customer_id", "type": "identifier", "format": "CUST-#####"},
        {"name": "full_name", "type": "name"},
        {"name": "email", "type": "email"},
        {"name": "age", "type": "integer", "minimum": 18, "maximum": 75},
        {"name": "gender", "type": "categorical", "allowed_values": ["Male", "Female", "Non-Binary", "Other"]},
        {"name": "subscription", "type": "categorical", "allowed_values": ["Free", "Basic", "Premium"]},
        {"name": "annual_income", "type": "float", "minimum": 20000.0, "maximum": 150000.0},
        {"name": "is_active", "type": "boolean"},
    ],
}


def _pretty(obj):
    return json.dumps(obj, indent=2, default=str)


def _skip():
    return gr.skip() if hasattr(gr, "skip") else gr.update()


# ============================================================
# HTML HELPERS
# ============================================================
def _banner(msg, kind="info"):
    icon = {"ok": "✅", "warn": "⚠️", "bad": "❌", "info": "ℹ️"}[kind]
    return f'<div class="sd-banner sd-{kind}"><span>{icon}</span><div>{msg}</div></div>'


NAV_HTML = """
<div class="sd-nav">
  <div class="sd-brand"><span class="sd-logo">🧬</span> Synthetic Data Studio</div>
  <div class="sd-nav-right">Schema&nbsp;→&nbsp;Dataset&nbsp;→&nbsp;Download</div>
</div>"""

HERO_HTML = """
<div class="sd-hero">
  <h1>Generate realistic synthetic datasets<br>from a simple schema</h1>
  <p>Upload your schema, choose how many rows you need, and get a verified, ready-to-use dataset in seconds.</p>
  <div class="sd-steps">
    <div><b>1</b><span>Upload schema</span></div>
    <div><b>2</b><span>Set row count</span></div>
    <div><b>3</b><span>Generate &amp; verify</span></div>
    <div><b>4</b><span>Browse &amp; download</span></div>
  </div>
</div>"""

FOOTER_HTML = """
<div class="sd-footer">Synthetic Data Studio · Schema analysis · Constraint &amp; LLM engine ·
Automatic verification &amp; refinement</div>"""

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root{--sd-primary:#4f46e5;--sd-primary2:#7c3aed;--sd-text:#0f172a;--sd-muted:#64748b;
      --sd-surface:#ffffff;--sd-border:#e2e8f0;--sd-bg:#f4f6fb;--sd-soft:#eef2ff;}
.dark{--sd-text:#e5e7eb;--sd-muted:#94a3b8;--sd-surface:#111827;--sd-border:#263041;
      --sd-bg:#0a0f1d;--sd-soft:#1b2140;}
.gradio-container{font-family:'Inter',system-ui,sans-serif !important;background:var(--sd-bg) !important;
      max-width:1180px !important;margin:0 auto !important;padding-bottom:10px !important;}
footer{display:none !important;}

/* nav + hero */
.sd-nav{display:flex;justify-content:space-between;align-items:center;padding:14px 4px 10px;}
.sd-brand{font-weight:800;font-size:19px;color:var(--sd-text);letter-spacing:-.2px;}
.sd-logo{display:inline-grid;place-items:center;width:34px;height:34px;border-radius:10px;margin-right:8px;
      background:linear-gradient(135deg,var(--sd-primary),var(--sd-primary2));font-size:18px;}
.sd-nav-right{font-size:13px;color:var(--sd-muted);font-weight:500;}
.sd-hero{background:linear-gradient(135deg,#4338ca 0%,#6d28d9 55%,#be185d 120%);color:#fff;border-radius:20px;
      padding:38px 40px 30px;margin:6px 0 18px;box-shadow:0 10px 30px rgba(79,70,229,.25);}
.sd-hero h1{margin:0 0 10px;font-size:34px;line-height:1.18;font-weight:800;letter-spacing:-.6px;color:#fff;}
.sd-hero p{margin:0 0 22px;font-size:15.5px;opacity:.92;max-width:640px;color:#fff;}
.sd-steps{display:flex;flex-wrap:wrap;gap:10px;}
.sd-steps div{display:flex;align-items:center;gap:9px;background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.22);
      padding:8px 14px;border-radius:999px;font-size:13px;font-weight:600;}
.sd-steps b{display:grid;place-items:center;width:22px;height:22px;border-radius:50%;background:#fff;color:#4338ca;font-size:12px;}

/* cards */
.sd-card{background:var(--sd-surface) !important;border:1px solid var(--sd-border) !important;border-radius:16px !important;
      padding:20px !important;box-shadow:0 1px 3px rgba(15,23,42,.05);}
.sd-title{font-size:15px;font-weight:700;color:var(--sd-text);margin:0 0 2px;}
.sd-sub{font-size:12.5px;color:var(--sd-muted);margin:0 0 12px;}
.sd-btn{background:linear-gradient(135deg,var(--sd-primary),var(--sd-primary2)) !important;color:#fff !important;border:none !important;
      font-weight:700 !important;font-size:15px !important;border-radius:12px !important;
      box-shadow:0 6px 16px rgba(79,70,229,.32) !important;}
.sd-btn:hover{filter:brightness(1.07);transform:translateY(-1px);}
.sd-btn2{border-radius:10px !important;font-weight:600 !important;}

/* banners */
.sd-banner{display:flex;gap:10px;align-items:flex-start;padding:11px 14px;border-radius:12px;font-size:13.5px;
      border:1px solid transparent;color:var(--sd-text);}
.sd-ok{background:rgba(34,197,94,.12);border-color:rgba(34,197,94,.4);}
.sd-warn{background:rgba(245,158,11,.13);border-color:rgba(245,158,11,.45);}
.sd-bad{background:rgba(239,68,68,.12);border-color:rgba(239,68,68,.45);}
.sd-info{background:var(--sd-soft);border-color:rgba(99,102,241,.35);}

/* schema chips */
.sd-chips{display:flex;flex-wrap:wrap;gap:8px;}
.sd-chip{border:1px solid var(--sd-border);background:var(--sd-bg);border-radius:10px;padding:7px 11px;font-size:12.5px;color:var(--sd-text);}
.sd-chip b{font-weight:700;}
.sd-chip i{font-style:normal;margin-left:7px;padding:1px 8px;border-radius:999px;font-size:11px;font-weight:700;
      background:var(--sd-soft);color:var(--sd-primary);}
.sd-chip small{display:block;color:var(--sd-muted);font-size:11px;margin-top:2px;}

/* KPIs */
.sd-kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;}
.sd-kpi{background:var(--sd-surface);border:1px solid var(--sd-border);border-radius:14px;padding:16px 18px;
      display:flex;align-items:center;gap:14px;}
.sd-kpi .v{font-size:23px;font-weight:800;color:var(--sd-text);line-height:1.1;}
.sd-kpi .l{font-size:12px;color:var(--sd-muted);font-weight:500;margin-top:3px;}
.sd-ring{--p:0;--c:#16a34a;width:62px;height:62px;border-radius:50%;flex:none;position:relative;display:grid;place-items:center;
      background:conic-gradient(var(--c) calc(var(--p)*1%),var(--sd-border) 0);}
.sd-ring::before{content:"";position:absolute;inset:7px;border-radius:50%;background:var(--sd-surface);}
.sd-ring span{position:relative;font-weight:800;font-size:17px;color:var(--sd-text);}
.sd-pill{display:inline-block;padding:2px 10px;border-radius:999px;font-size:11px;font-weight:700;margin-top:4px;}
.sd-pill.sd-ok{color:#15803d;} .sd-pill.sd-warn{color:#b45309;} .sd-pill.sd-bad{color:#b91c1c;}
.sd-pageinfo{font-size:13px;color:var(--sd-muted);padding:8px 2px;}
.sd-status{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;}
.sd-status div{border:1px solid var(--sd-border);border-radius:10px;padding:9px 12px;font-size:11.5px;color:var(--sd-muted);}
.sd-status b{display:block;font-size:13.5px;color:var(--sd-text);margin-bottom:2px;}
.sd-footer{text-align:center;font-size:12px;color:var(--sd-muted);padding:18px 0 6px;}
.tab-nav button,.tabs button{font-weight:600 !important;}
"""


def _schema_summary_html(schema_text):
    try:
        report, _ = _make_spec(_parse_schema_text(schema_text))
    except Exception as e:
        return _banner(str(e), "bad")
    chips = []
    for c in report["columns"]:
        extra = []
        if c.get("min") is not None or c.get("max") is not None:
            extra.append(f'{c.get("min", "")} – {c.get("max", "")}')
        av = c.get("allowed_values")
        if isinstance(av, list):
            extra.append(", ".join(map(str, av[:4])) + ("…" if len(av) > 4 else ""))
        if c.get("format"):
            extra.append(c["format"])
        if c.get("unique"):
            extra.append("unique")
        small = f"<small>{' · '.join(extra)}</small>" if extra else ""
        chips.append(f'<div class="sd-chip"><b>{c["name"]}</b><i>{c["type"]}</i>{small}</div>')
    engine_badge = '<span style="margin-left:8px;padding:2px 8px;border-radius:12px;font-size:12px;background:#8b5cf6;color:white">🤖 LLM extracted</span>' if report.get("extracted_by") == "llm" else '<span style="margin-left:8px;padding:2px 8px;border-radius:12px;font-size:12px;background:#3b82f6;color:white">⚡ Engine extracted</span>'
    return (_banner(f"Schema is valid — <b>{len(chips)}</b> columns detected. {engine_badge}", "ok")
            + f'<div class="sd-chips" style="margin-top:12px">{"".join(chips)}</div>')


def _kpi_html(res):
    a = res["after"]
    score = a["overall_score"]
    kind = {"PASS": "ok", "WARNING": "warn", "FAIL": "bad"}.get(a["status"], "info")
    color = {"ok": "#16a34a", "warn": "#d97706", "bad": "#dc2626", "info": "#6366f1"}[kind]
    passed = sum(1 for c in a["checks"].values() if c["status"] == "PASS")
    sub = ""
    if res["refined"] and res["before"]["overall_score"] != score:
        sub = f' <span style="font-weight:500;color:var(--sd-muted)">(was {res["before"]["overall_score"]:.0f})</span>'
    cards = [
        f'<div class="sd-kpi"><div class="sd-ring" style="--p:{score};--c:{color}"><span>{score:.0f}</span></div>'
        f'<div><div class="v">Quality</div><span class="sd-pill sd-{kind}">{a["status"]}</span>{sub}'
        f'<div style="margin-top:4px;font-size:11px;font-weight:600;color:{color}">{"🔓 Download Ready" if passed == len(a["checks"]) and a["status"] == "PASS" else "🔒 Download Locked"}</div></div></div>',
        f'<div class="sd-kpi"><div><div class="v">{res["rows"]:,}</div><div class="l">Rows generated</div></div></div>',
        f'<div class="sd-kpi"><div><div class="v">{res["cols"]}</div><div class="l">Columns</div></div></div>',
        f'<div class="sd-kpi"><div><div class="v">{passed}/{len(a["checks"])}</div><div class="l">Checks passed</div></div></div>',
        f'<div class="sd-kpi"><div><div class="v" style="font-size:16px">{res["engine_label"]}</div>'
        f'<div class="l">Engine · {res["seconds"]:.1f}s</div></div></div>',
    ]
    return f'<div class="sd-kpis">{"".join(cards)}</div>'


# ============================================================
# SCHEMA LOADING
# ============================================================
def _parse_schema_text(text):
    text = (text or "").strip()
    if not text:
        raise ValueError("Schema is empty — upload a file or paste your schema JSON.")
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON (line {e.lineno}, column {e.colno}): {e.msg}")


def _split_values(v):
    if isinstance(v, str):
        parts = [p.strip() for p in re.split(r"[|;,]", v) if p.strip()]
        return parts if parts else v
    return v


def load_schema_file(file_path):
    """Upload handler → (schema_editor_text, status_html)."""
    if not file_path:
        return _skip(), ""
    path = file_path if isinstance(file_path, str) else getattr(file_path, "name", str(file_path))
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".json":
            with open(path, "r", encoding="utf-8") as f:
                schema_obj = json.load(f)
        elif ext in (".txt", ".md"):
            with open(path, "r", encoding="utf-8") as f:
                raw = f.read().strip()
            try:
                schema_obj = json.loads(raw)
            except Exception:
                raise ValueError("The .txt/.md file must contain a valid JSON schema.")
        elif ext == ".csv":
            sdf = pd.read_csv(path)
            sdf = sdf.astype(object).where(sdf.notna(), None)
            schema_obj = []
            for rec in sdf.to_dict(orient="records"):
                rec = {k: v for k, v in rec.items() if v is not None}
                for key in ("allowed_values", "values", "categories"):
                    if key in rec:
                        rec[key] = _split_values(rec[key])
                schema_obj.append(rec)
        else:
            raise ValueError("Schema file must be .json, .csv, .txt or .md")
        return _pretty(schema_obj), _banner(f"Loaded <b>{os.path.basename(path)}</b>", "ok")
    except Exception as e:
        return _skip(), _banner(f"Could not read file: {e}", "bad")


def _make_spec(schema_obj):
    report = analyze_schema(schema_obj)
    if not report["columns"]:
        raise ValueError('No columns recognised. Use {"columns": [{"name": "...", "type": "..."}]}.')
    return report, build_generation_specification(report)


# ============================================================
# CORE PIPELINE  (generate → verify → refine → verify)
# ============================================================
def _generate_df(spec, n_rows, engine_mode, batch_size, log, progress_cb=None):
    use_llm = engine_mode.startswith("Ollama")
    if use_llm and not is_ollama_online():
        log("⚠️ Ollama is offline → falling back to the Constraint Engine (see the LLM Engine tab).")
        use_llm = False
    used_llm_ok = False
    records = []
    total_batches = math.ceil(n_rows / batch_size)
    for b in range(total_batches):
        cur = min(batch_size, n_rows - len(records))
        t = time.time()
        if use_llm:
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                recs = generate_batch(spec, cur, start_row=len(records) + 1)
            src = globals().get("LAST_BATCH_SOURCE", "ollama")
            used_llm_ok = used_llm_ok or src == "ollama"
            for ln in buf.getvalue().splitlines():
                if "⚠️" in ln:
                    log("   " + ln.strip())
        else:
            recs = [
                generate_fallback_row(spec["columns"], len(records) + i + 1)
                for i in range(cur)
            ]
            src = "constraint-engine"
        records.extend(recs[:cur])
        log(f"Batch {b + 1}/{total_batches}: {cur} rows via {src} ({time.time() - t:.1f}s)")
        if progress_cb:
            progress_cb(0.1 + 0.6 * (b + 1) / total_batches, f"Generating batch {b + 1}/{total_batches}")

    expected = [c["name"] for c in spec["columns"]]
    df = pd.DataFrame(records)
    for c in spec["columns"]:
        if c["name"] not in df.columns:
            df[c["name"]] = [generate_fallback_record(c, i + 1) for i in range(len(df))]
    df = df[expected]
    if use_llm:
        label = f"Ollama · {MODEL_NAME}" if used_llm_ok else "Ollama→Fallback"
    else:
        label = "Constraint Engine"
    return df, label


def run_pipeline(schema_obj, n_rows, engine_mode, batch_size, seed, auto_refine, log, progress_cb=None):
    """Full pipeline. Returns a dict with df, reports, fixes, etc."""
    t0 = time.time()
    n_rows, batch_size = int(n_rows), int(batch_size)
    if seed not in (None, ""):
        random.seed(int(seed))
        np.random.seed(int(seed))
        try:
            from faker import Faker as _F
            _F.seed(int(seed))
        except Exception:
            pass

    if progress_cb:
        progress_cb(0.02, "Analyzing schema")
    analysis, spec = _make_spec(schema_obj)
    log(f"Schema analysed: {len(analysis['columns'])} columns.")

    df, engine_label = _generate_df(spec, n_rows, engine_mode, batch_size, log, progress_cb)

    if progress_cb:
        progress_cb(0.75, "Verifying")
    before = verify_synthetic_dataset_v2(df, analysis, spec, n_rows)
    log(f"Verification (raw): {before['status']} — {before['overall_score']}/100")

    fixes, after = [], before
    if auto_refine:
        if progress_cb:
            progress_cb(0.85, "Refining")
        df, fixes = refine_synthetic_dataset(df, analysis, before)
        after = verify_synthetic_dataset_v2(df, analysis, spec, n_rows)
        log(f"Refinement applied {len(fixes)} fixes → {after['status']} — {after['overall_score']}/100")

        # Second-pass quality assurance if any checks failed
        if after["status"] != "PASS" and after.get("overall_score", 0) < 100:
            df, fixes2 = refine_synthetic_dataset(df, analysis, after)
            fixes.extend(fixes2)
            after = verify_synthetic_dataset_v2(df, analysis, spec, n_rows)
            log(f"Second-pass refinement → {after['status']} — {after['overall_score']}/100")

    # keep notebook globals in sync so the original cells keep working afterwards
    globals().update(
        synthetic_df=df, analysis_report=analysis, generation_specification=spec,
        num_rows=n_rows, verification_report=before, re_verification=after,
    )
    return {
        "df": df, "analysis": analysis, "spec": spec, "before": before, "after": after,
        "fixes": fixes, "refined": bool(auto_refine), "engine_label": engine_label,
        "rows": len(df), "cols": len(df.columns), "seconds": time.time() - t0,
    }



def _profile_df(df):
    rows = []
    for c in df.columns:
        s = df[c]
        is_num = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
        rows.append({
            "Column": c, "Dtype": str(s.dtype), "Unique": int(s.nunique()),
            "Missing": int(s.isna().sum()),
            "Min": round(float(s.min()), 2) if is_num else "",
            "Max": round(float(s.max()), 2) if is_num else "",
            "Mean": round(float(s.mean()), 2) if is_num else "",
            "Sample": ", ".join(map(str, s.head(3).tolist())),
        })
    return pd.DataFrame(rows)


def make_plot(df, col):
    if df is None or not isinstance(df, pd.DataFrame) or col is None or col not in df.columns:
        return None
    s = df[col].dropna()
    fig, ax = plt.subplots(figsize=(7, 3.6))
    color = "#6366f1"
    is_num = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
    if is_num and s.nunique() > 15:
        ax.hist(s, bins=min(30, max(8, int(np.sqrt(len(s))))), color=color, edgecolor="white")
        ax.set_ylabel("count")
    else:
        vc = s.astype(str).value_counts().head(15)
        ax.bar(vc.index, vc.values, color=color)
        ax.set_ylabel("count")
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_title(col, fontsize=12, fontweight="bold")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    fig.tight_layout()
    plt.close(fig)
    return fig



# ============================================================
# RESULT BROWSER (search · sort · paginate)
# ============================================================
_ORIG = "(original order)"


def _view(df, query, sort_col, order):
    out = df
    q = (query or "").strip()
    if q:
        mask = np.zeros(len(out), dtype=bool)
        for c in out.columns:
            mask |= out[c].astype(str).str.contains(q, case=False, regex=False).to_numpy()
        out = out[mask]
    if sort_col and sort_col != _ORIG and sort_col in out.columns:
        out = out.sort_values(sort_col, ascending=(order == "Ascending"), kind="stable")
    return out


def render_page(df, query, sort_col, order, page_size, page):
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame(), "", 1
    f = _view(df, query, sort_col, order)
    ps = int(page_size)
    total = len(f)
    pages = max(1, math.ceil(total / ps))
    page = min(max(1, int(page or 1)), pages)
    chunk = f.iloc[(page - 1) * ps: page * ps]
    a = (page - 1) * ps + 1 if total else 0
    b = min(page * ps, total)
    filt = f" · filtered from {len(df):,}" if total != len(df) else ""
    info = f'<div class="sd-pageinfo">Showing <b>{a:,}–{b:,}</b> of <b>{total:,}</b> rows{filt} · page {page} of {pages}</div>'
    return chunk, info, page


def reset_page(df, query, sort_col, order, page_size):
    return render_page(df, query, sort_col, order, page_size, 1)


def go_prev(df, query, sort_col, order, page_size, page):
    return render_page(df, query, sort_col, order, page_size, int(page or 1) - 1)


def go_next(df, query, sort_col, order, page_size, page):
    return render_page(df, query, sort_col, order, page_size, int(page or 1) + 1)


# ============================================================
# GENERATE HANDLER
# ============================================================
def _fail(msg):
    return (_banner(msg, "bad"), gr.Column(visible=False), "", gr.Group(visible=False), pd.DataFrame(), "", 1, None, None, None, None,
            gr.Dropdown(choices=[_ORIG], value=_ORIG), pd.DataFrame(),
            gr.Dropdown(choices=[], value=None), None, pd.DataFrame(), "", "")


def generate_ui(schema_text, n_rows, engine, batch_size, seed, auto_refine, model, page_size,
                progress=gr.Progress()):
    global MODEL_NAME
    if model and str(model).strip():
        MODEL_NAME = str(model).strip()
    try:
        n_rows = int(n_rows)
    except Exception:
        return _fail("Please enter a valid number of rows.")
    if not 1 <= n_rows <= MAX_ROWS:
        return _fail(f"Number of rows must be between 1 and {MAX_ROWS:,}.")

    engine_mode = engine
    if engine.startswith("Auto"):
        engine_mode = "Ollama LLM" if is_ollama_online() else "Constraint Engine"

    log_lines = []
    log = log_lines.append
    try:
        schema_obj = _parse_schema_text(schema_text)
        res = run_pipeline(schema_obj, n_rows, engine_mode, batch_size, seed, auto_refine, log,
                           progress_cb=lambda f, d: progress(f, desc=d))
    except Exception as e:
        return _fail(f"{type(e).__name__}: {e}")

    progress(0.95, desc="Preparing downloads")
    df = res["df"]
    name = "dataset"
    if isinstance(schema_obj, dict):
        name = re.sub(r"[^A-Za-z0-9_-]+", "_", str(schema_obj.get("dataset_name", "dataset"))) or "dataset"
    base = os.path.join(OUTPUT_DIR, f"{name}_synthetic")
    df.to_csv(base + ".csv", index=False)
    df.to_json(base + ".json", orient="records", indent=2)
    xlsx = None
    try:
        df.to_excel(base + ".xlsx", index=False)
        xlsx = base + ".xlsx"
    except Exception:
        log("Excel export skipped (openpyxl not available).")
    df.to_csv("refined_synthetic_dataset.csv", index=False)
    with open("analysis_report.json", "w") as f:
        json.dump(res["analysis"], f, indent=4, default=str)

    checks = []
    for k, v in res["after"]["checks"].items():
        b = res["before"]["checks"].get(k, {}).get("status", "—")
        row = {"Check": k.replace("_", " ").title(),
               "Before refinement": ("✅ " if b == "PASS" else "❌ ") + b,
               "Final": ("✅ " if v["status"] == "PASS" else "❌ ") + v["status"],
               "Details": v["details"]}
        checks.append(row)
    fixes_txt = "\n".join(f"✓ {x}" for x in res["fixes"]) if res["fixes"] else (
        "No refinement fixes were needed." if res["refined"] else "Auto-refinement is turned off.")

    chunk, info, page = render_page(df, "", _ORIG, "Ascending", page_size, 1)
    cols = list(df.columns)
    progress(1.0, desc="Done")

    # QUALITY GATE: Only show & allow download if ALL verification tests pass
    after_report = res["after"]
    checks_dict = after_report.get("checks", {})
    failed_checks = [k.replace("_", " ").title() for k, v in checks_dict.items() if v.get("status") != "PASS"]
    all_passed = (len(failed_checks) == 0) and (after_report.get("status") == "PASS" or after_report.get("overall_score", 0) >= 90)

    if all_passed:
        status = _banner(f"🎉 <b>All verification checks PASSED ({after_report.get('overall_score', 100):.0f}/100)</b> — Generated {len(df):,} rows in {res['seconds']:.1f}s. Dataset verified and unlocked for download below.", "ok")
        csv_out, json_out, xlsx_out = base + ".csv", base + ".json", xlsx
        dl_card_update = gr.Group(visible=True)
    else:
        status = _banner(f"⛔ <b>Quality Gate Blocked ({len(failed_checks)} check(s) failed):</b> {', '.join(failed_checks)}. Dataset downloads are locked until all verification tests pass. Review the <b>Quality report</b> tab below.", "bad")
        csv_out, json_out, xlsx_out = None, None, None
        dl_card_update = gr.Group(visible=False)

    return (status, gr.Column(visible=True), _kpi_html(res), dl_card_update, chunk, info, page,
            csv_out, json_out, xlsx_out, df,
            gr.Dropdown(choices=[_ORIG] + cols, value=_ORIG), _profile_df(df),
            gr.Dropdown(choices=cols, value=cols[0]), make_plot(df, cols[0]),
            pd.DataFrame(checks), fixes_txt, "\n".join(log_lines))


# ============================================================
# BUILD THE UI
# ============================================================
THEME = gr.themes.Soft(primary_hue="indigo", secondary_hue="violet", neutral_hue="slate",
                       font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"])
_launch_has_theme = "theme" in inspect.signature(gr.Blocks.launch).parameters
_blocks_kwargs = {} if _launch_has_theme else {"theme": THEME, "css": CSS}
_df_extra = {"max_height": 520} if "max_height" in inspect.signature(gr.DataFrame.__init__).parameters else {}


def _card_head(title, sub):
    return gr.HTML(f'<div class="sd-title">{title}</div><div class="sd-sub">{sub}</div>')


with gr.Blocks(title="Synthetic Data Studio", **_blocks_kwargs) as web_ui:
    gr.HTML(NAV_HTML)
    gr.HTML(HERO_HTML)
    df_state = gr.State(None)

    # ------------------------------------------------ INPUT SECTION
    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            with gr.Group(elem_classes="sd-card"):
                _card_head("1 · Your schema", "Upload a schema file or paste it as JSON.")
                with gr.Tabs():
                    with gr.Tab("📁 Upload file"):
                        schema_file = gr.File(label="Schema file (.json, .csv, .txt, .md)",
                                              file_types=[".json", ".csv", ".txt", ".md"], type="filepath")
                        upload_status = gr.HTML()
                    with gr.Tab("✏️ Paste JSON"):
                        schema_box = gr.Code(value=_pretty(DEFAULT_SCHEMA), language="json",
                                             label="Schema JSON", lines=14)
        with gr.Column(scale=4):
            with gr.Group(elem_classes="sd-card"):
                _card_head("2 · Dataset size", f"How many rows should be generated? (1 – {MAX_ROWS:,})")
                rows_in = gr.Number(value=100, precision=0, minimum=1, maximum=MAX_ROWS, label="Number of rows")
                with gr.Accordion("⚙️ Advanced settings", open=False):
                    engine_rd = gr.Radio(
                        ["Auto (Ollama if running)", "Constraint Engine", "Ollama LLM"],
                        value="Auto (Ollama if running)", label="Generation engine",
                        info="Constraint Engine is instant. Ollama LLM is slower but more realistic.")
                    batch_sl = gr.Slider(5, 25, value=10, step=1, label="Batch size (LLM)")
                    seed_in = gr.Number(value=None, precision=0, label="Random seed (optional)")
                    refine_cb = gr.Checkbox(value=True, label="Auto-refine & repair after verification")
                gen_btn = gr.Button("⚡  Generate dataset", variant="primary", size="lg", elem_classes="sd-btn")
                status_html = gr.HTML(_banner("Upload a schema, enter the rows and press <b>Generate</b>.", "info"))

    with gr.Group(elem_classes="sd-card"):
        _card_head("Schema preview", "Columns detected from your schema — updates as you edit.")
        schema_summary = gr.HTML(_schema_summary_html(_pretty(DEFAULT_SCHEMA)))

    model_dd = gr.State(MODEL_NAME)

    # ------------------------------------------------ RESULTS
    with gr.Column(visible=False) as results_col:
        kpi_html = gr.HTML()
        with gr.Group(visible=False, elem_classes="sd-card") as download_card:
            _card_head("⬇️ Download", "Your generated dataset in the format you need.")
            download_notice = gr.HTML()
            with gr.Row():
                csv_dl = gr.File(label="CSV", interactive=False)
                json_dl = gr.File(label="JSON", interactive=False)
                xlsx_dl = gr.File(label="Excel", interactive=False)
        with gr.Group(elem_classes="sd-card"):
            with gr.Tabs():
                with gr.Tab("📋 Browse data"):
                    with gr.Row():
                        search_in = gr.Textbox(label="Search", placeholder="Type to filter rows…", scale=3)
                        sort_dd = gr.Dropdown(choices=[_ORIG], value=_ORIG, label="Sort by", scale=2)
                        order_rd = gr.Radio(["Ascending", "Descending"], value="Ascending", label="Order", scale=2)
                        psize_dd = gr.Dropdown(choices=[10, 25, 50, 100], value=25, label="Rows per page", scale=1)
                    table = gr.DataFrame(interactive=False, wrap=True, label="Dataset", **_df_extra)
                    with gr.Row():
                        page_info = gr.HTML()
                        prev_btn = gr.Button("◀ Previous", scale=0, elem_classes="sd-btn2")
                        page_num = gr.Number(value=1, precision=0, minimum=1, label="Page", scale=0, min_width=110)
                        next_btn = gr.Button("Next ▶", scale=0, elem_classes="sd-btn2")
                with gr.Tab("📊 Column insights"):
                    profile_tbl = gr.DataFrame(label="Column profile", interactive=False, wrap=True)
                    col_pick = gr.Dropdown(choices=[], label="Chart a column")
                    col_plot = gr.Plot(label="Distribution")
                with gr.Tab("✅ Quality report"):
                    checks_tbl = gr.DataFrame(label="Verification checks", interactive=False, wrap=True)
                    fixes_box = gr.Textbox(label="Automatic fixes applied", lines=5, interactive=False)
                    log_box = gr.Textbox(label="Run log", lines=6, interactive=False)

    gr.HTML(FOOTER_HTML)

    # ------------------------------------------------ EVENTS
    schema_file.upload(load_schema_file, schema_file, [schema_box, upload_status])
    schema_box.change(_schema_summary_html, schema_box, schema_summary)

    gen_btn.click(
        generate_ui,
        [schema_box, rows_in, engine_rd, batch_sl, seed_in, refine_cb, model_dd, psize_dd],
        [status_html, results_col, kpi_html, download_card, table, page_info, page_num, csv_dl, json_dl, xlsx_dl,
         df_state, sort_dd, profile_tbl, col_pick, col_plot, checks_tbl, fixes_box, log_box],
    )

    _browse_in = [df_state, search_in, sort_dd, order_rd, psize_dd]
    _browse_out = [table, page_info, page_num]
    search_in.input(reset_page, _browse_in, _browse_out)
    sort_dd.input(reset_page, _browse_in, _browse_out)
    order_rd.input(reset_page, _browse_in, _browse_out)
    psize_dd.input(reset_page, _browse_in, _browse_out)
    prev_btn.click(go_prev, _browse_in + [page_num], _browse_out)
    next_btn.click(go_next, _browse_in + [page_num], _browse_out)
    page_num.submit(render_page, _browse_in + [page_num], _browse_out)
    col_pick.input(make_plot, [df_state, col_pick], col_plot)

    # Ollama UI buttons removed

# ------------------------------------------------ LAUNCH
print("Launching dashboard ... (open the public link below, or use the inline view)")
_launch_kwargs = {"share": True, "debug": False}
if _launch_has_theme:
    _launch_kwargs.update(theme=THEME, css=CSS)
web_ui.launch(**_launch_kwargs)

